# Open Science Colab v1.3 — Hardened Provenance-Preserving Decision Invariance under Homomorphic Transformation

**Research question:** Under a fixed operational policy, how robustly does a real HEIR/OpenFHE CKKS transformation preserve model outputs and downstream decisions, and which part of any discrepancy is attributable to float32 model export versus homomorphic evaluation?

## Why v1.3 exists
The completed v1.0 PILOT established technical feasibility on the original Wisconsin Breast Cancer split. v1.2 corrected the main methodological weakness of the interim v1.1 design by preserving the **original v1.0 split exactly**: 341 training cases, 114 calibration cases, and the original 114 held-out cases. v1.3 retains that scientific design unchanged and adds integrity/reproducibility hardening before the definitive PILOT/freeze.

The primary locked test is not an external cohort; it is the **provenance-preserving v1.0 held-out subset**, which was not used for v1.0 PILOT model fitting or calibration metrics. This choice sacrifices rare-event power in exchange for a cleaner development/test boundary.

The hardened design has six safeguards:

1. **Provenance-preserving locked test.** The v1.0 split is reproduced exactly and checked against pre-recorded SHA-256 identities for the full allocation and locked-test IDs.
2. **Transitive code integrity.** Scientific and confirmatory fingerprints recursively include notebook-defined helper functions referenced by the frozen analysis code, closing the gap left by root-function-only bytecode hashes.
3. **Environment provenance.** The freeze records Python implementation/full version, platform, OS, architecture, libc/loader evidence, and a strict compatibility contract; descriptive environment differences are reported without silently changing the study.
4. **Error decomposition and mechanistic stress.** Every HE result separates float64→float32 export error, incremental HE error, and total end-to-end error, while the boundary stress set carries calibration-only plausibility diagnostics.
5. **Sensitivity and multiplicity controls.** The notebook freezes multiple near-boundary cutoffs, multiple operational-threshold pairs, exact rare-event bounds, Holm-adjusted H2 p-values, paired bootstrap intervals, and a paired sign-flip randomization test for H4.
6. **Conservative interpretation.** H1 is a strict coexistence flag; H3 remains diagnostic; H2 and H4 are pre-specified endpoint-specific analyses without an omnibus FWER claim across heterogeneous H1–H4 endpoints; zero events are reported with exact finite-sample upper bounds.

## Two-phase workflow

### Phase A — PILOT / calibration
- reproduce the v1.0 development/locked-test allocation without materializing the locked-test features or labels;
- train and calibrate only on the v1.0 development data;
- execute the two pre-specified HE configurations on the complete calibration split;
- execute the same binaries on the calibration-derived boundary stress set;
- freeze the model, split provenance, policy, robustness specifications, code-integrity manifest, environment contract, binaries, dependencies, and runtime libraries;
- export the complete bundle and preserve the printed freeze SHA-256.

### Phase B — CONFIRMATORY / locked test
In a **fresh Colab session**, restore the v1.3 bundle, set `STUDY_MODE="CONFIRMATORY"`, provide the preserved freeze digest, and run from the first cell. No model retraining, configuration selection, boundary tuning, plausibility-threshold tuning, policy-threshold tuning, or analysis-code modification is allowed after locked-test access.

## Interpretation guardrails
- `SIMULATED_METHOD_VALIDATION` is methodology-development evidence only.
- `HEIR_REAL_HE_PILOT_CALIBRATION` is calibration/feasibility evidence only.
- `HEIR_BOUNDARY_STRESS_DIAGNOSTIC` is mechanistic robustness evidence only and must **not** be reported as a population event rate.
- Only `HEIR_REAL_HE_CONFIRMATORY` supports locked-test confirmatory claims.
- The locked test is provenance-preserving, **not an external validation cohort**.
- Zero observed disagreements imply an exact finite-sample upper bound, not universal invariance.
- H3 is a diagnostic ranking property of `error / operational_margin`; no multiplicity-adjusted reject/accept claim is made for H3.
- H2 and H4 are separately pre-specified confirmatory endpoints; the study does not claim omnibus family-wise error control across the heterogeneous H1–H4 set.


## v1.3 integrity-hardening revision summary

This revision preserves the v1.2 scientific design and adds reproducibility/integrity controls before the definitive PILOT. It does **not** change the dataset, split, hypotheses, operational thresholds, HE configurations, boundary-stress deltas, or primary statistical endpoints.

Key changes:
- new project root and study version; v1.0–v1.2 artifacts remain intact;
- retains the exact v1.0 60% / 20% / 20% provenance-preserving split and its immutable SHA-256 checks;
- replaces root-function-only analysis fingerprints with a **transitive notebook-function manifest** and bundle SHA-256;
- freezes a richer environment provenance snapshot (Python implementation/full version, OS/platform, architecture, libc and `ldd` evidence) while keeping a clearly defined strict compatibility contract;
- upgrades the confirmatory freeze schema to v6;
- explicitly documents the multiplicity scope: H2 has Holm control across configurations, H3 is diagnostic, H4 is a separate pre-specified paired numerical endpoint, and no omnibus FWER claim is made across heterogeneous H1–H4 endpoints;
- adds an explicit open-science metadata warning when repository URL or license has not yet been supplied;
- retains rare-event bounds, boundary-stress plausibility checks, error decomposition, H4 bootstrap/sign-flip randomization, and all v1.2 provenance safeguards unchanged.


## 0. Global study configuration


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata as importlib_metadata
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

PROJECT_NAME = "decision-invariance-heir-v1-3"
STUDY_VERSION = "1.3.0"

AUTHOR = "Larissa de Oliveira Figueira"
AFFILIATION = "FACTI"
REPOSITORY_URL = None  # Set before public release; not required for experimental execution.
LICENSE_CODE = None     # Set before public release; license choice is not inferred by the notebook.

# ------------------------------------------------------------------
# USER-CONTROLLED PHASE SWITCH
# ------------------------------------------------------------------
STUDY_MODE = "PILOT"  # PILOT or CONFIRMATORY
RESET_PILOT_ROOT_ON_START = True
RESTORE_BUNDLE_ZIP = None
EXPECTED_CONFIRMATORY_FREEZE_SHA256 = None

# Provenance-preserving split: reproduce the original v1.0 allocation exactly.
# The v1.0 PILOT used seeds 42/43 with a 60/20/20 train/calibration/locked-test split.
SEED = 42
TRAIN_FRACTION = 0.60
CALIBRATION_FRACTION = 0.20
TEST_FRACTION = 0.20
V1_0_SPLIT_REFERENCE_SHA256 = "072a3beac08a6df0eab0e7d0441ce015fb59ba32705a363d31251a6cd27ed681"
V1_0_LOCKED_TEST_IDS_SHA256 = "01a94ef6037a0a946162894026bdba111ba44aa1e1bb5a08e3ab1218c0273295"
CLASSIFICATION_THRESHOLD = 0.0

OPERATIONAL_LOW_PROB = 0.30
OPERATIONAL_HIGH_PROB = 0.70
POLICY_SENSITIVITY_PROB_PAIRS = [
    (0.25, 0.75),
    (0.30, 0.70),  # primary
    (0.35, 0.65),
]

def logit(p):
    return math.log(p / (1.0 - p))

OPERATIONAL_LOW_THRESHOLD = logit(OPERATIONAL_LOW_PROB)
OPERATIONAL_HIGH_THRESHOLD = logit(OPERATIONAL_HIGH_PROB)
NEAR_BOUNDARY_QUANTILE = 0.10
NEAR_BOUNDARY_SENSITIVITY_QUANTILES = [0.05, 0.10, 0.20]
SIMULATED_SIGMAS = [0.01, 0.03, 0.10, 0.30, 1.00]

# Mechanistic boundary stress set. Distances are in plaintext logit-score units.
BOUNDARY_STRESS_DELTAS = [1e-8, 3e-8, 1e-7, 3e-7, 1e-6, 3e-6, 1e-5, 3e-5, 1e-4]
BOUNDARY_STRESS_ANCHORS_PER_BOUNDARY = 4
BOUNDARY_STRESS_ID_BASE = 1_000_000
STRESS_PLAUSIBILITY_QUANTILE = 0.99
STRESS_KNN_K = 5

CI_LEVEL = 0.95
FAMILYWISE_ALPHA = 0.05
BOOTSTRAP_REPS_PILOT = 2000
BOOTSTRAP_REPS_CONFIRMATORY = 10000
H4_RANDOMIZATION_REPS = 20000
BOOTSTRAP_REPS = BOOTSTRAP_REPS_CONFIRMATORY if STUDY_MODE == "CONFIRMATORY" else BOOTSTRAP_REPS_PILOT

NUMPY_VERSION = "2.1.3"
HEIR_PY_VERSION = "2026.9.1"
HEIR_PY_SPEC = f"heir_py[openfhe]=={HEIR_PY_VERSION}"
HEIR_RELEASE_COMMIT = "5eaf0be77f9c07428e37d21cf4d7313d1295872e"
HEIR_REPO = "https://github.com/google/heir.git"
OPENFHE_REPO = "https://github.com/openfheorg/openfhe-development.git"

CKKS_RUNTIME_REPEATS = 3
CKKS_PILOT_SAMPLE_LIMIT = None
HE_EXECUTION_TIMEOUT_SECONDS = 3600

# Keep the two configurations that already completed the full v1.0 real-HE pilot.
# Robustness comes from error decomposition, preservation of the original held-out
# cohort, and a mechanistic boundary challenge rather than post-hoc HE-parameter searching.
HE_CONFIGURATION_SPECS = [
    {
        "configuration_id": "ckks_reference_f55_s45",
        "min_slot_count": 32,
        "first_mod_bits": 55,
        "scaling_mod_bits": 45,
        "role": "reference_precision",
    },
    {
        "configuration_id": "ckks_reduced_f50_s40",
        "min_slot_count": 32,
        "first_mod_bits": 50,
        "scaling_mod_bits": 40,
        "role": "reduced_precision",
    },
]
AUTO_FREEZE_AFTER_PILOT = True

random.seed(SEED)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
ROOT = Path("/content") / PROJECT_NAME

if STUDY_MODE == "PILOT" and RESET_PILOT_ROOT_ON_START and ROOT.exists():
    shutil.rmtree(ROOT)

if STUDY_MODE == "CONFIRMATORY" and RESTORE_BUNDLE_ZIP:
    restore_path = Path(RESTORE_BUNDLE_ZIP)
    if not restore_path.is_file():
        raise FileNotFoundError(f"RESTORE_BUNDLE_ZIP not found: {restore_path}")
    freeze_target = ROOT / "protocol" / "confirmatory_freeze.json"
    if not freeze_target.exists():
        ROOT.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(restore_path), str(ROOT))
        print("Restored pilot bundle into", ROOT)

RUN_DIR = ROOT / "runs" / RUN_ID
LOG_DIR = RUN_DIR / "logs"
CONFIRMATORY_FREEZE_PATH = ROOT / "protocol" / "confirmatory_freeze.json"
REGISTRY_DIR = ROOT / "protocol" / "he_configurations"

for folder in [
    ROOT / "protocol", ROOT / "environment", ROOT / "data", ROOT / "models",
    ROOT / "artifacts", ROOT / "results", ROOT / "figures", ROOT / "src",
    REGISTRY_DIR, RUN_DIR, LOG_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_NAME)
print("Study version:", STUDY_VERSION)
print("Study mode:", STUDY_MODE)
print("Run ID:", RUN_ID)
print("Split fractions:", TRAIN_FRACTION, CALIBRATION_FRACTION, TEST_FRACTION)
print("Operational thresholds:", OPERATIONAL_LOW_THRESHOLD, OPERATIONAL_HIGH_THRESHOLD)
print("Pre-specified HE configurations:", [c["configuration_id"] for c in HE_CONFIGURATION_SPECS])


## 1. Confirmatory-mode guard


In [ ]:
def is_full_git_sha(value):
    return bool(re.fullmatch(r"[0-9a-fA-F]{40}", str(value)))

if not is_full_git_sha(HEIR_RELEASE_COMMIT):
    raise ValueError("HEIR_RELEASE_COMMIT must be an exact 40-character SHA.")
if "==" not in HEIR_PY_SPEC:
    raise ValueError("HEIR_PY_SPEC must be explicitly version pinned.")
if STUDY_MODE not in {"PILOT", "CONFIRMATORY"}:
    raise ValueError("STUDY_MODE must be PILOT or CONFIRMATORY.")
if not math.isclose(TRAIN_FRACTION + CALIBRATION_FRACTION + TEST_FRACTION, 1.0, rel_tol=0, abs_tol=1e-12):
    raise ValueError("Train/calibration/test fractions must sum to 1.")
if TRAIN_FRACTION <= 0 or CALIBRATION_FRACTION <= 0 or TEST_FRACTION <= 0:
    raise ValueError("All split fractions must be positive.")
if len(HE_CONFIGURATION_SPECS) != 2:
    raise ValueError("v1.3 pre-specifies exactly two HE configurations for paired H4 analysis.")
if NEAR_BOUNDARY_QUANTILE not in NEAR_BOUNDARY_SENSITIVITY_QUANTILES:
    raise ValueError("Primary near-boundary quantile must be present in the sensitivity set.")
if (OPERATIONAL_LOW_PROB, OPERATIONAL_HIGH_PROB) not in POLICY_SENSITIVITY_PROB_PAIRS:
    raise ValueError("Primary operational policy must be present in the sensitivity set.")
if any(d <= 0 for d in BOUNDARY_STRESS_DELTAS) or sorted(BOUNDARY_STRESS_DELTAS) != BOUNDARY_STRESS_DELTAS:
    raise ValueError("Boundary-stress deltas must be positive and sorted.")
ids = [c["configuration_id"] for c in HE_CONFIGURATION_SPECS]
if len(set(ids)) != len(ids):
    raise ValueError("HE configuration IDs must be unique.")
for cfg in HE_CONFIGURATION_SPECS:
    if cfg["min_slot_count"] < 1 or cfg["first_mod_bits"] < 1 or cfg["scaling_mod_bits"] < 1:
        raise ValueError(f"Invalid HE configuration: {cfg}")

print("Version/study-mode guard passed.")
print("Pinned heir_py:", HEIR_PY_SPEC)
print("Pinned HEIR source commit:", HEIR_RELEASE_COMMIT)

existing_freeze = CONFIRMATORY_FREEZE_PATH
if STUDY_MODE == "PILOT" and existing_freeze.exists():
    raise RuntimeError("This project is already frozen. Start a clean PILOT root or switch to CONFIRMATORY.")
if STUDY_MODE == "CONFIRMATORY":
    if not existing_freeze.is_file() or not EXPECTED_CONFIRMATORY_FREEZE_SHA256:
        raise RuntimeError(
            "Restore the complete pilot bundle and set EXPECTED_CONFIRMATORY_FREEZE_SHA256 "
            "to the independently preserved digest."
        )
    actual = hashlib.sha256(existing_freeze.read_bytes()).hexdigest()
    if actual != EXPECTED_CONFIRMATORY_FREEZE_SHA256:
        raise RuntimeError("Freeze digest mismatch; stopping before dependency installation.")


In [ ]:
# Integrity helpers are defined before experimental results are interpreted.
import inspect

def canonical_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)

def digest_json(value):
    return hashlib.sha256(canonical_json(value).encode()).hexdigest()

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json_exclusive(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("x", encoding="utf-8") as stream:
        stream.write(canonical_json(value) + "\n")

def artifact_path(relative):
    path = (ROOT / relative).resolve()
    if not path.is_relative_to(ROOT.resolve()):
        raise ValueError("Artifact path escapes the project directory.")
    return path

def verify_artifacts(records):
    for relative, expected in records.items():
        path = artifact_path(relative)
        if not path.is_file() or sha256_file(path) != expected:
            raise RuntimeError(f"Missing or modified frozen artifact: {relative}")

def checked_id(value):
    if not isinstance(value, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,100}", value):
        raise ValueError("Invalid configuration identifier.")
    return value

def code_fingerprint(function):
    import types
    def describe(code):
        constants = [describe(v) if isinstance(v, types.CodeType) else repr(v) for v in code.co_consts]
        return {
            "bytecode": code.co_code.hex(), "constants": constants,
            "names": list(code.co_names), "varnames": list(code.co_varnames),
            "argcount": code.co_argcount, "kwonlyargcount": code.co_kwonlyargcount,
        }
    return digest_json(describe(function.__code__))

def transitive_code_manifest(root_functions, namespace=None):
    """Fingerprint root functions and every notebook-defined global helper they reference transitively."""
    namespace = globals() if namespace is None else namespace
    records = {}
    visiting = set()

    def collect(function):
        if not inspect.isfunction(function):
            raise TypeError(f"Expected a Python function, got {type(function)!r}")
        key = function.__qualname__
        if key in records or key in visiting:
            return
        visiting.add(key)
        records[key] = code_fingerprint(function)
        for global_name in function.__code__.co_names:
            candidate = namespace.get(global_name)
            if inspect.isfunction(candidate) and getattr(candidate, "__globals__", None) is namespace:
                collect(candidate)
        visiting.remove(key)

    for function in root_functions:
        collect(function)
    ordered = dict(sorted(records.items()))
    return {
        "roots": [fn.__qualname__ for fn in root_functions],
        "functions": ordered,
        "bundle_sha256": digest_json(ordered),
    }

def runtime_environment_provenance():
    def first_line(command):
        try:
            proc = subprocess.run(command, capture_output=True, text=True, check=False)
            text = (proc.stdout or proc.stderr or "").strip()
            return text.splitlines()[0] if text else None
        except Exception:
            return None

    libc_name, libc_version = platform.libc_ver()
    return {
        "python_implementation": platform.python_implementation(),
        "python_full_version": platform.python_version(),
        "python_version_info": list(sys.version_info[:3]),
        "python_build": list(platform.python_build()),
        "python_executable": str(Path(sys.executable).resolve()),
        "platform": platform.platform(),
        "system": platform.system(),
        "release": platform.release(),
        "machine": platform.machine(),
        "processor": platform.processor(),
        "libc": {"name": libc_name or None, "version": libc_version or None},
        "ldd_version_line": first_line(["ldd", "--version"]),
    }

def environment_compatibility_contract(provenance=None):
    provenance = runtime_environment_provenance() if provenance is None else provenance
    return {
        "python_implementation": provenance["python_implementation"],
        "python_major_minor": provenance["python_version_info"][:2],
        "system": provenance["system"],
        "machine": provenance["machine"],
    }

def scientific_snapshot():
    dataset_bytes = X.to_csv(index=True, float_format="%.17g").encode()
    target_bytes = y.to_csv(index=True).encode()
    challenge_bytes = BOUNDARY_STRESS_X.to_csv(index=False, float_format="%.17g").encode()
    challenge_meta_bytes = BOUNDARY_STRESS_META.to_csv(index=False, float_format="%.17g").encode()
    analysis_functions = [
        apply_operational_policy, operational_margin, evaluate_decision_invariance,
        clopper_pearson_interval, clopper_pearson_upper_bound,
        paired_bootstrap_evidence, boundary_cutoff_sensitivity,
        policy_threshold_sensitivity, holm_adjust, paired_signflip_randomization_test,
        float32_export_score, build_boundary_stress_set,
    ]
    return {
        "dataset_sha256": hashlib.sha256(dataset_bytes + b"\nTARGET\n" + target_bytes).hexdigest(),
        "boundary_stress_sha256": hashlib.sha256(challenge_bytes + b"\nMETA\n" + challenge_meta_bytes).hexdigest(),
        "train_ids": [int(v) for v in train_idx],
        "calibration_ids": [int(v) for v in cal_idx],
        "test_ids": [int(v) for v in test_idx],
        "feature_names": list(X.columns),
        "weights": [float(v) for v in w_raw],
        "bias": float(b_raw),
        "analysis_code": transitive_code_manifest(analysis_functions),
        "seed": SEED,
        "split_fractions": [TRAIN_FRACTION, CALIBRATION_FRACTION, TEST_FRACTION],
        "split_identity_sha256": split_identity_sha256,
        "v1_0_split_reference_sha256": V1_0_SPLIT_REFERENCE_SHA256,
        "locked_test_ids_sha256": locked_test_ids_sha256,
        "policy": {
            "classification_threshold": CLASSIFICATION_THRESHOLD,
            "low": OPERATIONAL_LOW_THRESHOLD,
            "high": OPERATIONAL_HIGH_THRESHOLD,
            "near_boundary_cutoff": NEAR_BOUNDARY_CUTOFF,
            "near_boundary_quantile": NEAR_BOUNDARY_QUANTILE,
            "near_boundary_sensitivity_cutoffs": NEAR_BOUNDARY_CUTOFFS,
            "policy_sensitivity_probability_pairs": POLICY_SENSITIVITY_PROB_PAIRS,
        },
        "boundary_stress": {
            "deltas": BOUNDARY_STRESS_DELTAS,
            "anchors_per_boundary": BOUNDARY_STRESS_ANCHORS_PER_BOUNDARY,
            "construction": "minimum L2 shift in standardized feature space along logistic-regression normal vector",
            "interpretation": "mechanistic diagnostic only; not population incidence",
            "plausibility": BOUNDARY_STRESS_PLAUSIBILITY,
        },
        "he_configuration_specs": HE_CONFIGURATION_SPECS,
        "analysis": {
            "confidence_level": CI_LEVEL,
            "familywise_alpha": FAMILYWISE_ALPHA,
            "bootstrap_reps_confirmatory": BOOTSTRAP_REPS_CONFIRMATORY,
            "h4_randomization_reps": H4_RANDOMIZATION_REPS,
            "repeats": CKKS_RUNTIME_REPEATS,
            "primary_repeat": 0,
            "repeat_rule": "repeat zero primary; repetitions characterize numerical stability and are not independent samples",
            "H1_rule": (
                "strict coexistence flag: classification_disagreement_count == 0 AND "
                "operational_action_disagreement_count > 0; aggregate fidelity reported separately without an arbitrary margin"
            ),
            "H2_rule": "primary 0.10 calibration-quantile near/far Fisher test; Holm-adjusted across configurations; 0.05/0.20 are sensitivity-only",
            "H3_rule": (
                "diagnostic AUC(error/margin) vs AUC(raw error) only when disagreement labels contain both classes; "
                "no reject/accept multiplicity claim"
            ),
            "H4_rule": "paired B-minus-A bootstrap and sign-flip randomization; primary endpoint is incremental HE absolute error relative to float32-export plaintext",
            "multiplicity_scope": {
                "H2": "Holm family-wise adjustment across the two configuration-specific primary Fisher tests",
                "H3": "diagnostic/descriptive; no reject/accept multiplicity claim",
                "H4": "separate pre-specified paired numerical endpoint with bootstrap interval and sign-flip randomization",
                "omnibus_H1_to_H4_FWER_claim": False,
            },
        },
    }

def dependency_versions():
    from packaging.requirements import Requirement
    from packaging.utils import canonicalize_name
    roots = [
        "numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "joblib",
        "colorama", "pybind11", "packaging", "jedi>=0.16", "heir_py[openfhe]",
    ]
    queue = [Requirement(v) for v in roots]
    visited, result = set(), {}
    while queue:
        req = queue.pop()
        name = canonicalize_name(req.name)
        extras = tuple(sorted(req.extras))
        key = (name, extras)
        if key in visited:
            continue
        visited.add(key)
        dist = importlib_metadata.distribution(req.name)
        result[name] = dist.version
        for raw in dist.requires or []:
            child = Requirement(raw)
            if child.marker is None or any(child.marker.evaluate({"extra": extra}) for extra in ("", *extras)):
                queue.append(child)
    return dict(sorted(result.items()))


## 2. Freeze the protocol before result inspection


In [ ]:
protocol = {
    "project_name": PROJECT_NAME,
    "study_version": STUDY_VERSION,
    "study_mode": STUDY_MODE,
    "research_question": (
        "Under a fixed three-action policy, how robustly does a real CKKS/HEIR transformation "
        "preserve outputs and decisions, and which discrepancies arise from float32 export versus HE evaluation?"
    ),
    "hypotheses": {
        "H1": (
            "Strict empirical coexistence flag: determine whether binary predictions are exactly preserved "
            "while at least one downstream operational action changes; also report aggregate binary-fidelity "
            "metrics without imposing an application-independent non-inferiority margin."
        ),
        "H2": "When operational disagreements occur, they are more concentrated near operational decision boundaries.",
        "H3": (
            "When disagreements occur, error normalized by operational margin ranks disagreement more strongly "
            "than raw error; this is a diagnostic ranking hypothesis, not a separate reject/accept inferential family."
        ),
        "H4": "The two pre-specified CKKS configurations differ in incremental HE numerical error after removing the common float32-export component.",
    },
    "data_design": {
        "train_fraction": TRAIN_FRACTION,
        "calibration_fraction": CALIBRATION_FRACTION,
        "locked_test_fraction": TEST_FRACTION,
        "split_seed_train_test": SEED,
        "split_seed_train_calibration": SEED + 1,
        "v1_0_split_reference_sha256": V1_0_SPLIT_REFERENCE_SHA256,
        "v1_0_locked_test_ids_sha256": V1_0_LOCKED_TEST_IDS_SHA256,
        "split_origin": "Exact reproduction of the original v1.0 60/20/20 allocation.",
        "locked_test_status": (
            "Primary confirmatory cohort is the original v1.0 held-out subset. It is not an external cohort and "
            "is not enlarged by re-splitting records previously used in v1.0 development."
        ),
        "locked_test_rule": "Locked features/labels are not materialized before confirmatory gate validation.",
    },
    "operational_policy": {
        "type": "three_action_policy",
        "primary_probability_pair": [OPERATIONAL_LOW_PROB, OPERATIONAL_HIGH_PROB],
        "sensitivity_probability_pairs": POLICY_SENSITIVITY_PROB_PAIRS,
        "low_logit_threshold": OPERATIONAL_LOW_THRESHOLD,
        "high_logit_threshold": OPERATIONAL_HIGH_THRESHOLD,
        "actions": ["ACTION_0", "REVIEW", "ACTION_1"],
    },
    "near_boundary_definition": {
        "primary_calibration_quantile": NEAR_BOUNDARY_QUANTILE,
        "sensitivity_quantiles": NEAR_BOUNDARY_SENSITIVITY_QUANTILES,
        "rule": "Distance to nearest operational boundary; every cutoff is calibrated on calibration only and frozen.",
    },
    "boundary_stress_design": {
        "deltas": BOUNDARY_STRESS_DELTAS,
        "anchors_per_boundary": BOUNDARY_STRESS_ANCHORS_PER_BOUNDARY,
        "construction": "minimum standardized-space shift along model normal to target both sides of each policy boundary",
        "plausibility_quantile": STRESS_PLAUSIBILITY_QUANTILE,
        "knn_k": STRESS_KNN_K,
        "plausibility_rule": (
            "Point must remain inside calibration marginal min/max and have both shrinkage-Mahalanobis distance "
            "and mean kNN distance at or below calibration-only q99 thresholds."
        ),
        "evidence_label": "HEIR_BOUNDARY_STRESS_DIAGNOSTIC",
        "population_inference_allowed": False,
    },
    "primary_metrics": [
        "output_mae", "he_mae_vs_float32_export", "classification_disagreement_rate",
        "operational_action_disagreement_rate", "near_boundary_operational_disagreement_rate",
    ],
    "statistical_evidence": {
        "exact_binomial_ci": True,
        "exact_one_sided_upper_bound": True,
        "paired_bootstrap_ci": True,
        "bootstrap_reps_pilot": BOOTSTRAP_REPS_PILOT,
        "bootstrap_reps_confirmatory": BOOTSTRAP_REPS_CONFIRMATORY,
        "H1": (
            "strict flag = zero binary-prediction disagreements AND at least one operational-action disagreement; "
            "aggregate accuracy/disagreement metrics are reported separately without an arbitrary non-inferiority margin"
        ),
        "H2": "one-sided Fisher exact test at primary boundary cutoff; Holm correction across configurations",
        "H3": "diagnostic bootstrap of AUC(error_to_margin_ratio) - AUC(raw_error); no reject/accept multiplicity claim",
        "H4": "paired bootstrap + sign-flip randomization on incremental HE error; total error is secondary",
        "multiplicity_scope": {
            "H2_family": "Holm-adjusted across the two configuration-specific primary Fisher tests",
            "H3": "diagnostic only; no reject/accept claim",
            "H4": "separate pre-specified paired numerical endpoint",
            "omnibus_H1_to_H4_FWER_claim": False,
            "interpretation": "Endpoint-specific claims only; the study does not advertise global FWER control across heterogeneous H1-H4 endpoints.",
        },
    },
    "he_configuration_specs": HE_CONFIGURATION_SPECS,
    "method_validation_sigmas": SIMULATED_SIGMAS,
    "seed": SEED,
    "warning": "SIMULATED_METHOD_VALIDATION and HEIR_BOUNDARY_STRESS_DIAGNOSTIC are not locked-test population evidence.",
    "validity_scope": {
        "dataset_scope": "single sklearn Wisconsin Breast Cancer benchmark",
        "workload_scope": "single affine logistic-regression score evaluated through HEIR/OpenFHE CKKS",
        "external_replication": "not part of this confirmatory family; required for stronger external-validity claims",
    },
    "infrastructure_pin": {
        "heir_py_version": HEIR_PY_VERSION,
        "heir_release_commit": HEIR_RELEASE_COMMIT,
        "openfhe_version_rule": "Derive the OpenFHE base release from the pinned HEIR MODULE.bazel.",
    },
}

protocol_path = ROOT / "protocol" / "protocol.json"
if STUDY_MODE == "PILOT":
    protocol_path.write_text(json.dumps(protocol, indent=2), encoding="utf-8")

print(json.dumps(protocol, indent=2))


## 3. Failure-aware external command runner


In [ ]:

def run_logged(cmd, stage, cwd=None, env=None):
    cmd = list(map(str, cmd))
    started = datetime.now(timezone.utc).isoformat()
    t0 = time.time()

    proc = subprocess.run(
        cmd,
        cwd=cwd,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    elapsed = time.time() - t0
    safe_stage = re.sub(r"[^A-Za-z0-9_.-]+", "_", stage)

    log_path = LOG_DIR / f"{safe_stage}.log"
    log_path.write_text(proc.stdout or "", encoding="utf-8")

    record = {
        "stage": stage,
        "command": cmd,
        "cwd": str(cwd) if cwd else None,
        "started_utc": started,
        "elapsed_seconds": elapsed,
        "return_code": proc.returncode,
        "log_path": str(log_path.relative_to(ROOT)),
        "status": "success" if proc.returncode == 0 else "failure",
    }

    record_path = LOG_DIR / f"{safe_stage}.json"
    record_path.write_text(json.dumps(record, indent=2), encoding="utf-8")

    print(f"[{record['status']}] {stage} ({elapsed:.1f}s)")
    if proc.stdout:
        print(proc.stdout[-3000:])

    if proc.returncode != 0:
        output_tail = (proc.stdout or "")[-5000:]
        raise RuntimeError(
            f"Stage '{stage}' failed. Full output saved to {log_path}.\n"
            f"Last subprocess output:\n{output_tail}"
        )

    return proc


## 4. Install Python dependencies


In [ ]:
# Install the scientific stack plus HEIR compiler/OpenFHE support.
# The experiment intentionally does NOT request the optional `heir_py[python]`
# frontend extra. The real-HE path uses heir-opt/heir-translate and generated C++.
import importlib

if sys.version_info < (3, 10):
    raise RuntimeError(
        f"heir_py {HEIR_PY_VERSION} requires Python >=3.10; "
        f"current interpreter is {sys.version.split()[0]}."
    )

python_requirements = [
    f"numpy=={NUMPY_VERSION}",
    "pandas",
    "matplotlib",
    "scikit-learn",
    "joblib",
    "scipy",
    "colorama",
    "packaging",
    "jedi>=0.16",
    HEIR_PY_SPEC,
]

loaded_versions = {
    module: getattr(sys.modules[module], "__version__", None)
    for module in ["numpy", "pandas", "scipy", "sklearn", "matplotlib"]
    if module in sys.modules
}

def pip_check_snapshot(stage):
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "check"],
        text=True, capture_output=True, check=False,
    )
    text = ((proc.stdout or "") + ("\n" + proc.stderr if proc.stderr else "")).strip()
    raw_lines = sorted({line.strip() for line in text.splitlines() if line.strip()})

    # `pip check` exits 0 when the environment is consistent and normally prints
    # the informational sentence "No broken requirements found.".  That sentence
    # is not a dependency problem and must never participate in set-difference
    # logic against an inconsistent baseline.
    issue_lines = raw_lines if proc.returncode != 0 else []
    record = {
        "stage": stage,
        "return_code": int(proc.returncode),
        "consistent": bool(proc.returncode == 0),
        "issues": issue_lines,
        "raw_output_lines": raw_lines,
    }
    (LOG_DIR / f"{stage}.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
    (LOG_DIR / f"{stage}.log").write_text(text + ("\n" if text else ""), encoding="utf-8")
    return record

# Colab images can contain pre-existing package-metadata inconsistencies that are
# unrelated to this experiment. Record them before installation so we can distinguish
# baseline issues from conflicts introduced by the study dependencies.
baseline_pip_check = pip_check_snapshot("python_dependency_consistency_baseline")

if STUDY_MODE == "CONFIRMATORY":
    install_freeze = json.loads((ROOT / "protocol" / "confirmatory_freeze.json").read_text())
    verify_artifacts(install_freeze["artifacts"])
    python_requirements = ["-r", str(ROOT / "environment" / "confirmatory_requirements.txt")]

run_logged(
    [sys.executable, "-m", "pip", "install", "--upgrade-strategy", "only-if-needed", *python_requirements],
    stage="install_python_dependencies",
)

# Any inconsistency newly introduced by this installation is fatal. Unchanged
# baseline Colab inconsistencies are recorded but do not invalidate the study stack.
final_pip_check = pip_check_snapshot("python_dependency_consistency_check")
baseline_issues = set(baseline_pip_check["issues"])
final_issues = set(final_pip_check["issues"])
new_inconsistencies = sorted(final_issues - baseline_issues)
resolved_inconsistencies = sorted(baseline_issues - final_issues)

if new_inconsistencies:
    raise RuntimeError(
        "The study dependency installation introduced new pip inconsistencies:\n"
        + "\n".join(new_inconsistencies)
        + f"\nFull records: {LOG_DIR / 'python_dependency_consistency_check.log'}"
    )

if final_pip_check["return_code"] != 0:
    print(
        "WARNING: only pre-existing environment inconsistencies remain after installation; "
        "they are recorded in",
        LOG_DIR / "python_dependency_consistency_check.log",
    )
elif resolved_inconsistencies:
    print(
        "Dependency check passed. Pre-existing inconsistency/inconsistencies were resolved:",
        *resolved_inconsistencies,
        sep="\n- ",
    )
else:
    print("Dependency check passed: no broken requirements found.")

importlib.invalidate_caches()

# Fresh-interpreter import/metadata preflight. No HEIR Python frontend import is
# required here; the study depends on the packaged compiler binaries.
preflight_code = f"""
import importlib
import importlib.metadata as metadata
import json
import shutil
import sys
modules = ["numpy", "pandas", "scipy", "sklearn", "matplotlib", "joblib",
           "colorama", "packaging", "pybind11", "jedi"]
for module in modules:
    importlib.import_module(module)
packages = ["numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "joblib",
            "colorama", "packaging", "pybind11", "pybind11_global", "jedi", "heir_py"]
versions = {{name: metadata.version(name) for name in packages}}
if versions["heir_py"] != "{HEIR_PY_VERSION}":
    raise RuntimeError("Resolved heir_py version differs from the study pin")
if versions["numpy"] != "{NUMPY_VERSION}":
    raise RuntimeError("Resolved NumPy version differs from the study pin")
for tool in ["heir-opt", "heir-translate"]:
    path = shutil.which(tool)
    if not path:
        raise RuntimeError(f"{{tool}} is not on PATH after heir_py installation")
print(json.dumps({{"python": sys.version, "packages": versions,
                  "heir_opt": shutil.which("heir-opt"),
                  "heir_translate": shutil.which("heir-translate")}}, indent=2))
"""
run_logged(
    [sys.executable, "-c", preflight_code],
    stage="python_dependency_import_preflight",
)

if importlib_metadata.version("heir_py") != HEIR_PY_VERSION:
    raise RuntimeError("Installed HEIR version differs from the study pin.")
if importlib_metadata.version("numpy") != NUMPY_VERSION:
    raise RuntimeError("Installed NumPy version differs from the study pin.")

for tool in ["heir-opt", "heir-translate"]:
    if not shutil.which(tool):
        raise RuntimeError(f"{tool} is unavailable after heir_py installation.")

distribution_names = {"sklearn": "scikit-learn"}
stale_modules = []
for module, loaded_version in loaded_versions.items():
    installed_version = importlib_metadata.version(distribution_names.get(module, module))
    if loaded_version is None or str(loaded_version) != installed_version:
        stale_modules.append(module)
if stale_modules:
    raise RuntimeError(
        "Dependencies were installed successfully, but this kernel still holds old modules: "
        + ", ".join(stale_modules)
        + ". Restart the Colab session, then run the notebook from the beginning."
    )

print("Python dependencies installed, dependency delta checked, and HEIR compiler binaries found.")



## 5. Resolve HEIR/OpenFHE compatibility and install OpenFHE

The notebook does **not** follow the current OpenFHE `main` branch.

Instead it:

1. installs the pinned stable `heir_py` release;
2. checks out the exact HEIR source commit associated with that release;
3. reads that release's `MODULE.bazel`;
4. extracts the OpenFHE dependency version expected by HEIR;
5. maps the BCR version to its upstream OpenFHE base release;
6. checks out that exact upstream release for the system CMake installation.

This avoids a silent mismatch between a frozen HEIR binary and a newer OpenFHE API.

HEIR's Bazel-specific OpenFHE patches are also inspected. If a pinned-release
patch touches non-Bazel/OpenFHE source files, the notebook refuses to continue
without explicit review rather than silently building an incompatible backend.


In [ ]:
if STUDY_MODE == "CONFIRMATORY":
    frozen_infra = json.loads((ROOT / "protocol" / "confirmatory_freeze.json").read_text())["infrastructure"]
    HEIR_OPENFHE_BCR_VERSION = frozen_infra["heir_expected_openfhe_bcr_version"]
    OPENFHE_GIT_REF = frozen_infra["openfhe_git_ref"]
    resolved_openfhe_sha = frozen_infra["openfhe_commit"]
    print("CONFIRMATORY: using frozen executables and bundled OpenFHE libraries; no rebuild.")
else:

    HEIR_SOURCE_DIR = Path("/content/heir-source")
    OPENFHE_SRC = Path("/content/openfhe-development")
    OPENFHE_BUILD = OPENFHE_SRC / "build"

    run_logged(["apt-get", "update", "-qq"], stage="apt_update")
    run_logged(
        [
            "apt-get", "install", "-y", "-qq",
            "build-essential", "cmake", "git", "libomp-dev"
        ],
        stage="install_system_build_dependencies",
    )

    # ---- exact HEIR source used by the pinned PyPI release ----
    if not HEIR_SOURCE_DIR.exists():
        run_logged(
            ["git", "clone", "--filter=blob:none", HEIR_REPO, str(HEIR_SOURCE_DIR)],
            stage="clone_heir_source",
        )

    run_logged(
        ["git", "fetch", "origin", HEIR_RELEASE_COMMIT],
        stage="fetch_pinned_heir_commit",
        cwd=HEIR_SOURCE_DIR,
    )
    run_logged(
        ["git", "checkout", "--detach", HEIR_RELEASE_COMMIT],
        stage="checkout_pinned_heir_commit",
        cwd=HEIR_SOURCE_DIR,
    )

    resolved_heir_source_sha = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=HEIR_SOURCE_DIR,
        text=True,
    ).strip()

    if resolved_heir_source_sha != HEIR_RELEASE_COMMIT:
        raise RuntimeError("Resolved HEIR source does not match the pinned release commit.")

    module_bazel = (HEIR_SOURCE_DIR / "MODULE.bazel").read_text(encoding="utf-8")

    openfhe_match = re.search(
        r'bazel_dep\(\s*name\s*=\s*"openfhe",\s*version\s*=\s*"([^"]+)"',
        module_bazel,
        flags=re.S,
    )
    if not openfhe_match:
        raise RuntimeError("Could not derive OpenFHE dependency from HEIR MODULE.bazel.")

    HEIR_OPENFHE_BCR_VERSION = openfhe_match.group(1)

    base_match = re.match(r"(\d+\.\d+\.\d+)", HEIR_OPENFHE_BCR_VERSION)
    if not base_match:
        raise RuntimeError(
            f"Could not map HEIR OpenFHE version {HEIR_OPENFHE_BCR_VERSION!r} "
            "to an upstream release."
        )

    OPENFHE_BASE_VERSION = base_match.group(1)
    OPENFHE_GIT_REF = f"v{OPENFHE_BASE_VERSION}"

    # Inspect pinned HEIR patches. Bazel/build-system-only patches are not needed
    # for the direct CMake system install used by the Python frontend.
    patch_records = []
    for patch_name in ["openfhe.patch", "openfhe_module.patch"]:
        patch_path = HEIR_SOURCE_DIR / "patches" / patch_name
        if not patch_path.exists():
            continue

        text = patch_path.read_text(encoding="utf-8", errors="replace")
        changed_files = re.findall(r"^\+\+\+ b/(.+)$", text, flags=re.M)

        source_relevant = [
            f for f in changed_files
            if not (
                f == "BUILD"
                or f == "MODULE.bazel"
                or f.endswith(".bzl")
                or f.endswith(".bazel")
            )
        ]

        patch_records.append({
            "patch": patch_name,
            "sha256": hashlib.sha256(patch_path.read_bytes()).hexdigest(),
            "changed_files": changed_files,
            "source_relevant_files": source_relevant,
        })

        if source_relevant:
            raise RuntimeError(
                f"Pinned HEIR patch {patch_name} touches OpenFHE source files "
                f"{source_relevant}. Manual compatibility review is required "
                "before a system CMake build."
            )

    # ---- exact upstream OpenFHE base release expected by this HEIR release ----
    if not OPENFHE_SRC.exists():
        run_logged(
            ["git", "clone", OPENFHE_REPO, str(OPENFHE_SRC)],
            stage="clone_openfhe",
        )

    # Remove prior build state before switching versions.
    run_logged(
        ["git", "reset", "--hard"],
        stage="reset_openfhe_worktree",
        cwd=OPENFHE_SRC,
    )
    run_logged(
        ["git", "clean", "-fdx"],
        stage="clean_openfhe_worktree",
        cwd=OPENFHE_SRC,
    )
    run_logged(
        ["git", "fetch", "--all", "--tags", "--force"],
        stage="fetch_openfhe",
        cwd=OPENFHE_SRC,
    )
    run_logged(
        ["git", "checkout", "--detach", OPENFHE_GIT_REF],
        stage="checkout_compatible_openfhe_release",
        cwd=OPENFHE_SRC,
    )

    resolved_openfhe_sha = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=OPENFHE_SRC,
        text=True,
    ).strip()

    OPENFHE_BUILD.mkdir(exist_ok=True)

    run_logged(
        [
            "cmake",
            "-S", str(OPENFHE_SRC),
            "-B", str(OPENFHE_BUILD),
            "-DCMAKE_BUILD_TYPE=Release",
            "-DBUILD_SHARED=ON",
            "-DBUILD_STATIC=OFF",
            "-DWITH_OPENMP=ON",
            "-DBUILD_UNITTESTS=OFF",
            "-DBUILD_EXAMPLES=OFF",
            "-DBUILD_BENCHMARKS=OFF",
        ],
        stage="configure_openfhe",
    )

    run_logged(
        ["cmake", "--build", str(OPENFHE_BUILD), "-j2"],
        stage="build_openfhe",
    )

    run_logged(
        ["cmake", "--install", str(OPENFHE_BUILD)],
        stage="install_openfhe",
    )

    run_logged(["ldconfig"], stage="ldconfig_after_openfhe")

    compatibility_record = {
        "heir_py_version_pinned": HEIR_PY_VERSION,
        "heir_release_commit": HEIR_RELEASE_COMMIT,
        "heir_source_commit_resolved": resolved_heir_source_sha,
        "heir_openfhe_bcr_version": HEIR_OPENFHE_BCR_VERSION,
        "openfhe_base_version": OPENFHE_BASE_VERSION,
        "openfhe_git_ref": OPENFHE_GIT_REF,
        "openfhe_commit_resolved": resolved_openfhe_sha,
        "heir_openfhe_patch_records": patch_records,
        "system_install_note": (
            "Direct OpenFHE CMake install; HEIR Bazel-only patch files are recorded "
            "but not applied."
        ),
    }

    (ROOT / "environment" / "heir_openfhe_compatibility.json").write_text(
        json.dumps(compatibility_record, indent=2),
        encoding="utf-8",
    )

    (ROOT / "environment" / "openfhe_resolved_commit.txt").write_text(
        resolved_openfhe_sha + "\n",
        encoding="utf-8",
    )

    print(json.dumps(compatibility_record, indent=2))


## 6. Capture the final environment and Git state


In [ ]:

environment = runtime_environment_provenance()
environment.update({
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "study_mode": STUDY_MODE,
    "study_version": STUDY_VERSION,
    "environment_contract": environment_compatibility_contract(),
    "heir_release_commit": HEIR_RELEASE_COMMIT,
    "heir_openfhe_bcr_version": HEIR_OPENFHE_BCR_VERSION,
    "openfhe_requested_ref": OPENFHE_GIT_REF,
    "openfhe_resolved_commit": resolved_openfhe_sha,
})

try:
    environment["heir_py_version"] = importlib_metadata.version("heir_py")
except importlib_metadata.PackageNotFoundError:
    environment["heir_py_version"] = None

environment_path = ROOT / "environment" / f"environment_{RUN_ID}.json"
environment_path.write_text(json.dumps(environment, indent=2), encoding="utf-8")

freeze = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True,
    text=True,
    check=True,
)
(ROOT / "environment" / "requirements.lock.txt").write_text(
    freeze.stdout,
    encoding="utf-8",
)

apt_snapshot = subprocess.run(
    ["dpkg-query", "-W", "-f=${Package}=${Version}\n"],
    capture_output=True,
    text=True,
    check=True,
)
(ROOT / "environment" / f"apt_packages_{RUN_ID}.txt").write_text(
    apt_snapshot.stdout,
    encoding="utf-8",
)

def capture_git_state(path):
    path = Path(path)
    try:
        commit = subprocess.check_output(
            ["git", "-C", str(path), "rev-parse", "HEAD"],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
        status = subprocess.check_output(
            ["git", "-C", str(path), "status", "--porcelain"],
            text=True,
            stderr=subprocess.DEVNULL,
        )
        return {
            "available": True,
            "commit": commit,
            "working_tree_clean": (status.strip() == ""),
        }
    except Exception:
        return {
            "available": False,
            "commit": None,
            "working_tree_clean": None,
        }

git_state = capture_git_state(ROOT)
(ROOT / "environment" / f"git_state_{RUN_ID}.json").write_text(
    json.dumps(git_state, indent=2),
    encoding="utf-8",
)

print(json.dumps(environment, indent=2))
print("Git state:", git_state)


## 7. HEIR tool provenance + BGV infrastructure smoke test


In [ ]:

heir_record = {}

for binary in ["heir-opt", "heir-translate"]:
    path = shutil.which(binary)
    heir_record[binary] = {"path": path}
    if path:
        proc = subprocess.run(
            [binary, "--version"],
            capture_output=True,
            text=True,
            check=False,
        )
        heir_record[binary]["version_output"] = (
            proc.stdout or proc.stderr
        ).strip()

(ROOT / "environment" / f"heir_tools_{RUN_ID}.json").write_text(
    json.dumps(heir_record, indent=2),
    encoding="utf-8",
)

print(json.dumps(heir_record, indent=2))


In [ ]:
if STUDY_MODE == "PILOT":
    compiler_smoke = {
        "test": "HEIR_compiler_binary_preflight",
        "python": sys.version,
        "numpy": importlib_metadata.version("numpy"),
        "heir_py": importlib_metadata.version("heir_py"),
        "tools": {},
    }

    for tool in ["heir-opt", "heir-translate"]:
        path = shutil.which(tool)
        if not path:
            raise RuntimeError(f"{tool} is not available after HEIR installation.")
        proc = subprocess.run(
            [path, "--help"],
            capture_output=True,
            text=True,
            check=False,
        )
        compiler_smoke["tools"][tool] = {
            "path": path,
            "return_code": int(proc.returncode),
            "help_sha256": hashlib.sha256(
                ((proc.stdout or "") + "\n" + (proc.stderr or "")).encode()
            ).hexdigest(),
        }
        if proc.returncode != 0:
            raise RuntimeError(f"{tool} --help failed with return code {proc.returncode}.")

    compiler_smoke["passed"] = True
    (RUN_DIR / "heir_compiler_binary_preflight.json").write_text(
        json.dumps(compiler_smoke, indent=2),
        encoding="utf-8",
    )
    print(json.dumps(compiler_smoke, indent=2))
    print("The end-to-end cryptographic validation is the real CKKS calibration pilot in section 18.")

else:
    print("Compiler smoke preflight skipped: confirmatory mode reuses frozen compiler-generated artifacts.")


## 8. CKKS capability record and configuration registry


In [ ]:
heir_opt = shutil.which("heir-opt")
ckks_capability = {
    "heir_opt_found": bool(heir_opt),
    "ckks_mentions_in_help": [],
    "pre_specified_configurations": HE_CONFIGURATION_SPECS,
}

if heir_opt:
    help_proc = subprocess.run(
        [heir_opt, "--help"], capture_output=True, text=True, check=False,
    )
    help_text = (help_proc.stdout or "") + "\n" + (help_proc.stderr or "")
    ckks_capability["ckks_mentions_in_help"] = [
        line.strip() for line in help_text.splitlines() if "ckks" in line.lower()
    ][:100]

(ROOT / "environment" / f"ckks_capability_{RUN_ID}.json").write_text(
    json.dumps(ckks_capability, indent=2), encoding="utf-8",
)
(ROOT / "protocol" / "he_configuration_specs.json").write_text(
    json.dumps(HE_CONFIGURATION_SPECS, indent=2), encoding="utf-8",
)

print("Pre-specified CKKS configurations recorded:", [c["configuration_id"] for c in HE_CONFIGURATION_SPECS])
print("CKKS help mentions:", len(ckks_capability["ckks_mentions_in_help"]))


## 9. Load dataset and create train / calibration / locked-test splits


In [ ]:
import numpy as np
import pandas as pd
import sklearn
import joblib

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.covariance import LedoitWolf
from sklearn.neighbors import NearestNeighbors
from scipy.stats import beta, fisher_exact

np.random.seed(SEED)

data = load_breast_cancer(as_frame=True)
X = data.data.copy()
y = data.target.copy()

all_indices = np.arange(len(X))

# Reproduce the original v1.0 split exactly. The locked-test allocation is not
# enlarged or re-randomized in v1.3.
dev_idx, test_idx = train_test_split(
    all_indices,
    test_size=TEST_FRACTION,
    random_state=SEED,
    stratify=y,
)

cal_fraction_within_dev = CALIBRATION_FRACTION / (
    TRAIN_FRACTION + CALIBRATION_FRACTION
)

train_idx, cal_idx = train_test_split(
    dev_idx,
    test_size=cal_fraction_within_dev,
    random_state=SEED + 1,
    stratify=y.iloc[dev_idx],
)

split_identity = {
    "train_ids": [int(v) for v in train_idx],
    "calibration_ids": [int(v) for v in cal_idx],
    "test_ids": [int(v) for v in test_idx],
}
split_identity_sha256 = digest_json(split_identity)
locked_test_ids_sha256 = hashlib.sha256(
    json.dumps(split_identity["test_ids"], separators=(",", ":")).encode()
).hexdigest()
if split_identity_sha256 != V1_0_SPLIT_REFERENCE_SHA256:
    raise RuntimeError("v1.3 did not reproduce the frozen v1.0 split allocation.")
if locked_test_ids_sha256 != V1_0_LOCKED_TEST_IDS_SHA256:
    raise RuntimeError("v1.3 locked-test IDs differ from the v1.0 held-out cohort.")

X_train = X.iloc[train_idx]
X_cal = X.iloc[cal_idx]
y_train = y.iloc[train_idx]
y_cal = y.iloc[cal_idx]

# Strict locked-test rule: keep only row indices until the confirmatory gate.
LOCKED_TEST_MATERIALIZED = False

split_record = pd.DataFrame({
    "row_index": np.concatenate([train_idx, cal_idx, test_idx]),
    "split": (
        ["train"] * len(train_idx)
        + ["calibration"] * len(cal_idx)
        + ["locked_test"] * len(test_idx)
    ),
})
split_path = ROOT / "data" / "split_indices.csv"
split_provenance_path = ROOT / "protocol" / "split_provenance.json"
split_provenance = {
    "origin": "exact_v1_0_split_reproduction",
    "train_test_seed": SEED,
    "train_calibration_seed": SEED + 1,
    "train_fraction": TRAIN_FRACTION,
    "calibration_fraction": CALIBRATION_FRACTION,
    "locked_test_fraction": TEST_FRACTION,
    "split_identity_sha256": split_identity_sha256,
    "expected_v1_0_split_identity_sha256": V1_0_SPLIT_REFERENCE_SHA256,
    "locked_test_ids_sha256": locked_test_ids_sha256,
    "expected_v1_0_locked_test_ids_sha256": V1_0_LOCKED_TEST_IDS_SHA256,
    "claim_scope": (
        "This is the original v1.0 held-out subset, not an external cohort. v1.3 does not create a larger "
        "test set by recycling records that appeared in v1.0 development data."
    ),
}
if STUDY_MODE == "PILOT":
    split_record.to_csv(split_path, index=False)
    split_provenance_path.write_text(json.dumps(split_provenance, indent=2), encoding="utf-8")
else:
    pd.testing.assert_frame_equal(pd.read_csv(split_path), split_record, check_dtype=False)
    if json.loads(split_provenance_path.read_text(encoding="utf-8")) != split_provenance:
        raise RuntimeError("Split provenance differs from the frozen PILOT.")

dataset_metadata = {
    "loader": "sklearn.datasets.load_breast_cancer",
    "sklearn_version": sklearn.__version__,
    "n_total": int(len(X)),
    "n_train": int(len(train_idx)),
    "n_calibration": int(len(cal_idx)),
    "n_locked_test": int(len(test_idx)),
    "n_features": int(X.shape[1]),
    "feature_names": list(X.columns),
    "seed_train_test": SEED,
    "seed_train_calibration": SEED + 1,
    "split_identity_sha256": split_identity_sha256,
    "locked_test_ids_sha256": locked_test_ids_sha256,
    "external_validation_cohort": False,
}

(ROOT / "data" / "dataset_metadata.json").write_text(
    json.dumps(dataset_metadata, indent=2),
    encoding="utf-8",
)

print(dataset_metadata)
print(json.dumps(split_provenance, indent=2))


## 10. Train plaintext model and export equivalent affine score


In [ ]:

if STUDY_MODE == "PILOT":
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, random_state=SEED)),
    ])
    model.fit(X_train, y_train)
    joblib.dump(model, ROOT / "models" / "plaintext_logistic_regression.joblib")
else:
    reference_freeze = json.loads((ROOT / "protocol" / "confirmatory_freeze.json").read_text())
    verify_artifacts(reference_freeze["artifacts"])
    model = joblib.load(ROOT / "models" / "plaintext_logistic_regression.joblib")

scaler = model.named_steps["scaler"]
clf = model.named_steps["clf"]

coef_scaled = clf.coef_.reshape(-1)
intercept_scaled = float(clf.intercept_[0])

# Equivalent affine model on raw input features.
w_raw = coef_scaled / scaler.scale_
b_raw = intercept_scaled - float(
    np.dot(coef_scaled, scaler.mean_ / scaler.scale_)
)

def affine_score(X_frame):
    return X_frame.to_numpy(dtype=float) @ w_raw + b_raw

# HEIR's current CKKS path in this study consumes f32 tensors. Freeze an explicit
# float32-export reference so HE-specific error can be separated from serialization.
w_export_f32 = np.asarray(w_raw, dtype=np.float32)
b_export_f32 = np.float32(b_raw)

def float32_export_score(X_frame):
    x32 = X_frame.to_numpy(dtype=np.float32) if hasattr(X_frame, "to_numpy") else np.asarray(X_frame, dtype=np.float32)
    return (x32 @ w_export_f32 + b_export_f32).astype(np.float64)

z_train = affine_score(X_train)
z_cal = affine_score(X_cal)
z_cal_export_f32 = float32_export_score(X_cal)

# Verify affine equivalence on development data only.
# The locked test remains untouched until the confirmatory gate.
max_development_equivalence_error = max(
    float(np.max(np.abs(model.decision_function(X_train) - z_train))),
    float(np.max(np.abs(model.decision_function(X_cal) - z_cal))),
)

assert max_development_equivalence_error <= 1e-10

if STUDY_MODE == "PILOT":
    np.savez(
        ROOT / "models" / "plaintext_affine_model.npz",
        weights=w_raw,
        bias=np.array([b_raw]),
        feature_names=np.array(X.columns, dtype=str),
    )

affine_metadata = {
    "decision_function": "z(x) = w^T x + b",
    "binary_classification_threshold": CLASSIFICATION_THRESHOLD,
    "max_development_pipeline_vs_affine_error": max_development_equivalence_error,
    "n_features": int(len(w_raw)),
    "bias": b_raw,
    "float32_export_calibration_mae_vs_float64": float(np.mean(np.abs(z_cal_export_f32 - z_cal))),
}

(ROOT / "models" / "plaintext_affine_model_metadata.json").write_text(
    json.dumps(affine_metadata, indent=2),
    encoding="utf-8",
)

print("Max development sklearn-vs-affine error:", max_development_equivalence_error)



## 11. Define and freeze the three-action operational policy

The downstream operational policy is intentionally distinct from the binary class prediction:

\[
\pi(z)=
\begin{cases}
\text{ACTION\_0}, & z < t_L\\
\text{REVIEW}, & t_L \le z < t_H\\
\text{ACTION\_1}, & z \ge t_H
\end{cases}
\]

with:

\[
t_L=\operatorname{logit}(0.30),\qquad
t_H=\operatorname{logit}(0.70).
\]

The **operational margin** is the distance to the nearest action-changing boundary:

\[
m_\pi(z)=\min(|z-t_L|,|z-t_H|).
\]

This is different from the binary class margin \(|z|\).


In [ ]:
ACTION_0 = "ACTION_0"
ACTION_REVIEW = "REVIEW"
ACTION_1 = "ACTION_1"

def apply_operational_policy(z):
    z = np.asarray(z, dtype=float)
    out = np.empty(z.shape, dtype=object)
    out[z < OPERATIONAL_LOW_THRESHOLD] = ACTION_0
    mid = (z >= OPERATIONAL_LOW_THRESHOLD) & (z < OPERATIONAL_HIGH_THRESHOLD)
    out[mid] = ACTION_REVIEW
    out[z >= OPERATIONAL_HIGH_THRESHOLD] = ACTION_1
    return out

def apply_operational_policy_thresholds(z, low_threshold, high_threshold):
    z = np.asarray(z, dtype=float)
    if not low_threshold < high_threshold:
        raise ValueError("low_threshold must be smaller than high_threshold")
    out = np.empty(z.shape, dtype=object)
    out[z < low_threshold] = ACTION_0
    mid = (z >= low_threshold) & (z < high_threshold)
    out[mid] = ACTION_REVIEW
    out[z >= high_threshold] = ACTION_1
    return out

def operational_margin(z):
    z = np.asarray(z, dtype=float)
    return np.minimum(np.abs(z - OPERATIONAL_LOW_THRESHOLD), np.abs(z - OPERATIONAL_HIGH_THRESHOLD))

calibration_margins = operational_margin(z_cal)
NEAR_BOUNDARY_CUTOFFS = {
    f"q{q:.2f}": float(np.quantile(calibration_margins, q))
    for q in NEAR_BOUNDARY_SENSITIVITY_QUANTILES
}
NEAR_BOUNDARY_CUTOFF = NEAR_BOUNDARY_CUTOFFS[f"q{NEAR_BOUNDARY_QUANTILE:.2f}"]

calibration_record = {
    "primary_near_boundary_quantile": NEAR_BOUNDARY_QUANTILE,
    "primary_near_boundary_cutoff": NEAR_BOUNDARY_CUTOFF,
    "sensitivity_cutoffs": NEAR_BOUNDARY_CUTOFFS,
    "n_calibration": int(len(z_cal)),
    "operational_low_threshold": OPERATIONAL_LOW_THRESHOLD,
    "operational_high_threshold": OPERATIONAL_HIGH_THRESHOLD,
    "policy_sensitivity_probability_pairs": POLICY_SENSITIVITY_PROB_PAIRS,
}

policy_path = ROOT / "protocol" / "frozen_operational_policy.json"
if STUDY_MODE == "PILOT":
    policy_path.write_text(json.dumps(calibration_record, indent=2), encoding="utf-8")
else:
    saved = json.loads(policy_path.read_text(encoding="utf-8"))
    if saved != calibration_record:
        raise RuntimeError("Operational policy/cutoffs differ from the frozen pilot.")

print(json.dumps(calibration_record, indent=2))


## 11.1 Calibration-derived mechanistic boundary stress set

The stress set is constructed **only from calibration anchors and the frozen plaintext model**. For each operational boundary, the nearest calibration anchors are shifted by the minimum Euclidean displacement in standardized feature space needed to reach pre-specified score distances on both sides of that boundary.

This set answers a mechanistic question — *can the complete float32+HE pipeline change an action when the plaintext score is extremely close to a policy boundary?* — and is therefore reported separately from locked-test incidence. Rows are correlated by construction and no population confidence interval is attached to their crossing rate.


In [ ]:
def build_boundary_stress_set(X_calibration, z_calibration):
    beta = np.asarray(coef_scaled, dtype=float)
    norm2 = float(np.dot(beta, beta))
    if not np.isfinite(norm2) or norm2 <= 0:
        raise RuntimeError("Invalid logistic-regression normal vector for boundary stress construction.")

    u_cal = scaler.transform(X_calibration)
    cal_rows = np.asarray(X_calibration.index, dtype=int)
    thresholds = [
        ("LOW", OPERATIONAL_LOW_THRESHOLD),
        ("HIGH", OPERATIONAL_HIGH_THRESHOLD),
    ]
    feature_min = X_calibration.min(axis=0).to_numpy(float)
    feature_max = X_calibration.max(axis=0).to_numpy(float)

    feature_rows = []
    meta_rows = []
    next_id = int(BOUNDARY_STRESS_ID_BASE)
    for boundary_name, threshold in thresholds:
        anchor_order = np.argsort(np.abs(np.asarray(z_calibration) - threshold))
        anchor_positions = anchor_order[:BOUNDARY_STRESS_ANCHORS_PER_BOUNDARY]
        for anchor_pos in anchor_positions:
            anchor_u = np.asarray(u_cal[anchor_pos], dtype=float)
            anchor_score = float(z_calibration[anchor_pos])
            anchor_row_index = int(cal_rows[anchor_pos])
            for side in (-1, 1):
                for delta in BOUNDARY_STRESS_DELTAS:
                    target_score = float(threshold + side * delta)
                    shift = ((target_score - anchor_score) / norm2) * beta
                    u_target = anchor_u + shift
                    x_target = scaler.inverse_transform(u_target.reshape(1, -1))[0]
                    x_frame = pd.DataFrame([x_target], columns=X.columns)
                    achieved = float(affine_score(x_frame)[0])
                    if abs(achieved - target_score) > 1e-9:
                        raise RuntimeError("Boundary stress construction failed to achieve the requested affine score.")
                    range_violations = int(np.sum((x_target < feature_min) | (x_target > feature_max)))
                    feature_rows.append(x_target)
                    meta_rows.append({
                        "sample_id": next_id,
                        "boundary": boundary_name,
                        "threshold": float(threshold),
                        "side": int(side),
                        "delta": float(delta),
                        "target_score": target_score,
                        "achieved_score_float64": achieved,
                        "anchor_row_index": anchor_row_index,
                        "anchor_score": anchor_score,
                        "standardized_shift_l2": float(np.linalg.norm(shift)),
                        "features_outside_calibration_minmax": range_violations,
                    })
                    next_id += 1

    features = pd.DataFrame(feature_rows, columns=X.columns)
    metadata = pd.DataFrame(meta_rows)

    # Joint-support diagnostics use calibration data only and are frozen before any
    # locked-test access. Ledoit-Wolf shrinkage stabilizes covariance estimation.
    u_stress = scaler.transform(features)
    lw = LedoitWolf().fit(u_cal)
    cal_mahal_sq = lw.mahalanobis(u_cal)
    stress_mahal_sq = lw.mahalanobis(u_stress)
    mahal_threshold = float(np.quantile(cal_mahal_sq, STRESS_PLAUSIBILITY_QUANTILE))

    k = min(int(STRESS_KNN_K), max(1, len(u_cal) - 1))
    nn = NearestNeighbors(n_neighbors=k + 1, metric="euclidean").fit(u_cal)
    cal_knn_all = nn.kneighbors(u_cal, n_neighbors=k + 1, return_distance=True)[0]
    cal_knn_mean = cal_knn_all[:, 1:].mean(axis=1)
    stress_knn_mean = nn.kneighbors(u_stress, n_neighbors=k, return_distance=True)[0].mean(axis=1)
    knn_threshold = float(np.quantile(cal_knn_mean, STRESS_PLAUSIBILITY_QUANTILE))

    metadata["mahalanobis_sq_shrinkage"] = stress_mahal_sq
    metadata["knn_mean_distance"] = stress_knn_mean
    metadata["within_calibration_marginal_minmax"] = metadata["features_outside_calibration_minmax"].eq(0)
    metadata["within_mahalanobis_q99"] = metadata["mahalanobis_sq_shrinkage"] <= mahal_threshold
    metadata["within_knn_q99"] = metadata["knn_mean_distance"] <= knn_threshold
    metadata["plausible_joint_support"] = (
        metadata["within_calibration_marginal_minmax"]
        & metadata["within_mahalanobis_q99"]
        & metadata["within_knn_q99"]
    )

    plausibility = {
        "calibration_only": True,
        "quantile": float(STRESS_PLAUSIBILITY_QUANTILE),
        "knn_k": int(k),
        "covariance_estimator": "LedoitWolf shrinkage covariance in standardized feature space",
        "mahalanobis_squared_threshold": mahal_threshold,
        "knn_mean_distance_threshold": knn_threshold,
        "plausibility_rule": (
            "inside calibration marginal min/max AND shrinkage Mahalanobis squared distance <= q99 "
            "AND mean kNN distance <= q99"
        ),
    }
    return features, metadata, plausibility

BOUNDARY_STRESS_X, BOUNDARY_STRESS_META, BOUNDARY_STRESS_PLAUSIBILITY = build_boundary_stress_set(X_cal, z_cal)
BOUNDARY_STRESS_IDS = BOUNDARY_STRESS_META["sample_id"].to_numpy(dtype=int)
BOUNDARY_STRESS_META["float32_export_score"] = float32_export_score(BOUNDARY_STRESS_X)
BOUNDARY_STRESS_META["serialization_error"] = np.abs(
    BOUNDARY_STRESS_META["float32_export_score"] - BOUNDARY_STRESS_META["achieved_score_float64"]
)

stress_features_path = ROOT / "data" / "boundary_stress_features.csv"
stress_meta_path = ROOT / "data" / "boundary_stress_metadata.csv"
stress_plausibility_path = ROOT / "protocol" / "boundary_stress_plausibility.json"
if STUDY_MODE == "PILOT":
    BOUNDARY_STRESS_X.to_csv(stress_features_path, index=False, float_format="%.17g")
    BOUNDARY_STRESS_META.to_csv(stress_meta_path, index=False, float_format="%.17g")
    stress_plausibility_path.write_text(json.dumps(BOUNDARY_STRESS_PLAUSIBILITY, indent=2), encoding="utf-8")
else:
    pd.testing.assert_frame_equal(
        pd.read_csv(stress_features_path, float_precision="round_trip"), BOUNDARY_STRESS_X, check_dtype=False, rtol=1e-12, atol=1e-12
    )
    pd.testing.assert_frame_equal(
        pd.read_csv(stress_meta_path, float_precision="round_trip"), BOUNDARY_STRESS_META, check_dtype=False, rtol=1e-12, atol=1e-12
    )
    if json.loads(stress_plausibility_path.read_text(encoding="utf-8")) != BOUNDARY_STRESS_PLAUSIBILITY:
        raise RuntimeError("Boundary-stress plausibility thresholds differ from the frozen PILOT.")

print({
    "boundary_stress_n": int(len(BOUNDARY_STRESS_X)),
    "deltas": BOUNDARY_STRESS_DELTAS,
    "anchors_per_boundary": BOUNDARY_STRESS_ANCHORS_PER_BOUNDARY,
    "max_target_achievement_error": float(np.max(np.abs(
        BOUNDARY_STRESS_META["target_score"] - BOUNDARY_STRESS_META["achieved_score_float64"]
    ))),
    "median_standardized_shift_l2": float(BOUNDARY_STRESS_META["standardized_shift_l2"].median()),
    "rows_with_any_feature_outside_calibration_minmax": int((BOUNDARY_STRESS_META["features_outside_calibration_minmax"] > 0).sum()),
    "plausible_joint_support_n": int(BOUNDARY_STRESS_META["plausible_joint_support"].sum()),
    "plausibility": BOUNDARY_STRESS_PLAUSIBILITY,
})


## 12. Statistical helper functions


In [ ]:
def clopper_pearson_interval(k, n, confidence=0.95):
    if n <= 0:
        return (None, None)
    alpha = 1.0 - confidence
    lower = 0.0 if k == 0 else float(beta.ppf(alpha / 2.0, k, n - k + 1))
    upper = 1.0 if k == n else float(beta.ppf(1.0 - alpha / 2.0, k + 1, n - k))
    return lower, upper

def clopper_pearson_upper_bound(k, n, confidence=0.95):
    if n <= 0:
        return None
    if k == n:
        return 1.0
    return float(beta.ppf(confidence, k + 1, n - k))

def percentile_ci(values, confidence=0.95):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return (None, None)
    alpha = 1.0 - confidence
    return float(np.quantile(values, alpha / 2.0)), float(np.quantile(values, 1.0 - alpha / 2.0))

def required_n_for_at_least_one_event(event_rate, probability=0.95):
    if not (0 < event_rate < 1) or not (0 < probability < 1):
        raise ValueError("event_rate and probability must be between 0 and 1.")
    return math.ceil(math.log(1.0 - probability) / math.log(1.0 - event_rate))

def event_rate_for_at_least_one(n, probability=0.95):
    if n <= 0 or not (0 < probability < 1):
        raise ValueError("n must be positive and probability between 0 and 1.")
    return float(1.0 - (1.0 - probability) ** (1.0 / n))

def holm_adjust(pvalues):
    arr = np.asarray(pvalues, dtype=float)
    if arr.ndim != 1:
        raise ValueError("pvalues must be one-dimensional")
    out = np.full(arr.shape, np.nan, dtype=float)
    valid = np.where(np.isfinite(arr))[0]
    if valid.size == 0:
        return out
    order = valid[np.argsort(arr[valid])]
    m = len(order)
    running = 0.0
    for rank, idx in enumerate(order):
        adjusted = min(1.0, (m - rank) * arr[idx])
        running = max(running, adjusted)
        out[idx] = running
    return out

def paired_signflip_randomization_test(differences, reps=20000, seed=42):
    diff = np.asarray(differences, dtype=float)
    diff = diff[np.isfinite(diff)]
    if diff.size == 0:
        return None
    observed = abs(float(np.mean(diff)))
    rng = np.random.default_rng(seed)
    extreme = 0
    for _ in range(int(reps)):
        signs = rng.choice(np.array([-1.0, 1.0]), size=diff.size)
        if abs(float(np.mean(diff * signs))) >= observed - 1e-18:
            extreme += 1
    return float((extreme + 1) / (int(reps) + 1))

rare_event_planning = {
    "locked_test_n": int(len(test_idx)),
    "event_rate_giving_95pct_probability_of_at_least_one_event": event_rate_for_at_least_one(len(test_idx), 0.95),
    "n_for_1pct_event_at_95pct_detection": required_n_for_at_least_one_event(0.01, 0.95),
    "n_for_0_5pct_event_at_95pct_detection": required_n_for_at_least_one_event(0.005, 0.95),
    "zero_event_exact_one_sided_upper_bound_95pct": clopper_pearson_upper_bound(0, len(test_idx), 0.95),
    "interpretation": (
        "The provenance-preserving locked test has limited power for rare disagreements; zero observed events "
        "must be reported with this finite-sample upper bound rather than as universal invariance."
    ),
}
rare_event_path = ROOT / "protocol" / "rare_event_planning.json"
if STUDY_MODE == "PILOT":
    rare_event_path.write_text(json.dumps(rare_event_planning, indent=2), encoding="utf-8")
else:
    if json.loads(rare_event_path.read_text(encoding="utf-8")) != rare_event_planning:
        raise RuntimeError("Rare-event planning differs from the frozen pilot.")
print(json.dumps(rare_event_planning, indent=2))


## 13. Decision-invariance evaluator


In [ ]:
def evaluate_decision_invariance(
    y_true, z_plain, z_transformed, near_boundary_cutoff,
    transformation_type, configuration_id,
):
    y_true = np.asarray(y_true)
    z_plain = np.asarray(z_plain, dtype=float)
    z_transformed = np.asarray(z_transformed, dtype=float)
    if z_plain.ndim != 1 or z_plain.size == 0 or y_true.shape != z_plain.shape:
        raise ValueError("Expected non-empty one-dimensional aligned labels and scores.")
    if not np.isfinite(z_plain).all() or not np.isfinite(z_transformed).all():
        raise ValueError("Scores must be finite.")
    if not np.isfinite(near_boundary_cutoff) or near_boundary_cutoff < 0:
        raise ValueError("Boundary cutoff must be finite and non-negative.")
    if z_plain.shape != z_transformed.shape:
        raise ValueError("Plaintext and transformed score shapes differ.")

    class_plain = (z_plain >= CLASSIFICATION_THRESHOLD).astype(int)
    class_transformed = (z_transformed >= CLASSIFICATION_THRESHOLD).astype(int)
    action_plain = apply_operational_policy(z_plain)
    action_transformed = apply_operational_policy(z_transformed)
    op_margin = operational_margin(z_plain)
    error = np.abs(z_transformed - z_plain)
    class_disagreement = class_plain != class_transformed
    action_disagreement = action_plain != action_transformed
    near = op_margin <= near_boundary_cutoff
    safe_margin = np.maximum(op_margin, np.finfo(float).eps)
    error_to_margin_ratio = error / safe_margin

    detail = pd.DataFrame({
        "y_true": y_true, "z_plain": z_plain, "z_transformed": z_transformed,
        "class_plain": class_plain, "class_transformed": class_transformed,
        "action_plain": action_plain, "action_transformed": action_transformed,
        "operational_margin": op_margin, "transformation_error": error,
        "error_to_margin_ratio": error_to_margin_ratio,
        "classification_disagreement": class_disagreement,
        "operational_action_disagreement": action_disagreement,
        "near_operational_boundary": near,
        "transformation_type": transformation_type,
        "configuration_id": configuration_id,
    })

    plain_acc = accuracy_score(y_true, class_plain)
    transformed_acc = accuracy_score(y_true, class_transformed)
    n_total = int(len(detail))
    k_action = int(action_disagreement.sum())
    k_class = int(class_disagreement.sum())
    action_ci = clopper_pearson_interval(k_action, n_total, CI_LEVEL)
    class_ci = clopper_pearson_interval(k_class, n_total, CI_LEVEL)
    near_n = int(near.sum())
    near_k = int((action_disagreement & near).sum())
    far_n = int((~near).sum())
    far_k = int((action_disagreement & (~near)).sum())
    near_ci = clopper_pearson_interval(near_k, near_n, CI_LEVEL) if near_n else (None, None)

    if k_action == 0:
        h2_status, fisher_odds_ratio, fisher_p = "NOT_ESTIMABLE_NO_OPERATIONAL_DISAGREEMENTS", None, None
    elif near_n == 0 or far_n == 0:
        h2_status, fisher_odds_ratio, fisher_p = "NOT_ESTIMABLE_EMPTY_STRATUM", None, None
    else:
        h2_status = "ESTIMABLE"
        odds, p = fisher_exact([[near_k, near_n - near_k], [far_k, far_n - far_k]], alternative="greater")
        fisher_odds_ratio = None if not np.isfinite(odds) else float(odds)
        fisher_p = float(p)

    if np.unique(action_disagreement.astype(int)).size == 2:
        h3_status = "ESTIMABLE_DIAGNOSTIC"
        labels = action_disagreement.astype(int)
        auc_raw_error = float(roc_auc_score(labels, error))
        auc_ratio = float(roc_auc_score(labels, error_to_margin_ratio))
    else:
        h3_status, auc_raw_error, auc_ratio = "NOT_ESTIMABLE_ONE_CLASS_DISAGREEMENT_LABEL", None, None

    metrics = {
        "transformation_type": transformation_type,
        "configuration_id": configuration_id,
        "n": n_total,
        "plaintext_binary_accuracy": float(plain_acc),
        "transformed_binary_accuracy": float(transformed_acc),
        "absolute_binary_accuracy_difference": float(abs(plain_acc - transformed_acc)),
        "output_mae": float(error.mean()),
        "output_median_abs_error": float(np.median(error)),
        "output_p95_abs_error": float(np.quantile(error, 0.95)),
        "output_p99_abs_error": float(np.quantile(error, 0.99)),
        "output_max_abs_error": float(error.max()),
        "classification_disagreement_rate": float(class_disagreement.mean()),
        "classification_disagreement_count": k_class,
        "classification_disagreement_ci_low": class_ci[0],
        "classification_disagreement_ci_high": class_ci[1],
        "classification_disagreement_upper_bound_one_sided": clopper_pearson_upper_bound(k_class, n_total, CI_LEVEL),
        "operational_action_disagreement_rate": float(action_disagreement.mean()),
        "operational_action_disagreement_count": k_action,
        "operational_action_disagreement_ci_low": action_ci[0],
        "operational_action_disagreement_ci_high": action_ci[1],
        "operational_action_disagreement_upper_bound_one_sided": clopper_pearson_upper_bound(k_action, n_total, CI_LEVEL),
        "near_boundary_n": near_n,
        "near_boundary_operational_disagreement_rate": float(action_disagreement[near].mean()) if near_n else None,
        "near_boundary_disagreement_ci_low": near_ci[0],
        "near_boundary_disagreement_ci_high": near_ci[1],
        "far_boundary_n": far_n,
        "far_boundary_operational_disagreement_rate": float(action_disagreement[~near].mean()) if far_n else None,
        "H2_status": h2_status,
        "H2_fisher_odds_ratio": fisher_odds_ratio,
        "H2_fisher_p_one_sided": fisher_p,
        "H3_status": h3_status,
        "H3_auc_raw_error": auc_raw_error,
        "H3_auc_error_to_margin_ratio": auc_ratio,
        "near_boundary_cutoff": float(near_boundary_cutoff),
    }
    return metrics, detail

def boundary_cutoff_sensitivity(detail, cutoff_map):
    rows = []
    disagreement = detail["operational_action_disagreement"].astype(bool).to_numpy()
    margins = detail["operational_margin"].to_numpy(float)
    for name, cutoff in cutoff_map.items():
        near = margins <= float(cutoff)
        near_n, far_n = int(near.sum()), int((~near).sum())
        near_k = int((disagreement & near).sum())
        far_k = int((disagreement & (~near)).sum())
        if disagreement.sum() == 0:
            status, odds, p = "NOT_ESTIMABLE_NO_OPERATIONAL_DISAGREEMENTS", None, None
        elif near_n == 0 or far_n == 0:
            status, odds, p = "NOT_ESTIMABLE_EMPTY_STRATUM", None, None
        else:
            o, pv = fisher_exact([[near_k, near_n-near_k], [far_k, far_n-far_k]], alternative="greater")
            status = "ESTIMABLE"
            odds = None if not np.isfinite(o) else float(o)
            p = float(pv)
        rows.append({
            "cutoff_name": name, "cutoff": float(cutoff), "near_n": near_n, "far_n": far_n,
            "near_disagreements": near_k, "far_disagreements": far_k,
            "status": status, "fisher_odds_ratio": odds, "fisher_p_one_sided": p,
            "is_primary": bool(name == f"q{NEAR_BOUNDARY_QUANTILE:.2f}"),
        })
    return pd.DataFrame(rows)

def policy_threshold_sensitivity(z_plain, z_transformed, probability_pairs):
    z_plain = np.asarray(z_plain, float)
    z_transformed = np.asarray(z_transformed, float)
    rows = []
    for low_prob, high_prob in probability_pairs:
        low_t, high_t = logit(low_prob), logit(high_prob)
        a = apply_operational_policy_thresholds(z_plain, low_t, high_t)
        b = apply_operational_policy_thresholds(z_transformed, low_t, high_t)
        disagreement = a != b
        k, n = int(disagreement.sum()), int(len(disagreement))
        ci = clopper_pearson_interval(k, n, CI_LEVEL)
        rows.append({
            "low_probability": float(low_prob), "high_probability": float(high_prob),
            "low_logit_threshold": float(low_t), "high_logit_threshold": float(high_t),
            "disagreement_count": k, "disagreement_rate": float(k/n),
            "ci_low": ci[0], "ci_high": ci[1],
            "upper_bound_one_sided": clopper_pearson_upper_bound(k, n, CI_LEVEL),
            "is_primary": bool(low_prob == OPERATIONAL_LOW_PROB and high_prob == OPERATIONAL_HIGH_PROB),
        })
    return pd.DataFrame(rows)


## 14. Deterministic unit tests for the evaluator


In [ ]:

def run_decision_invariance_unit_tests():
    # Use the actual three-action thresholds.
    tL = OPERATIONAL_LOW_THRESHOLD
    tH = OPERATIONAL_HIGH_THRESHOLD

    # Plain actions:
    # ACTION_0, REVIEW, REVIEW, ACTION_1, ACTION_1
    z_plain_unit = np.array([
        tL - 0.10,
        tL + 0.05,
        tH - 0.05,
        tH + 0.10,
        3.00,
    ])

    # Transformed actions:
    # REVIEW, ACTION_0, ACTION_1, ACTION_1, ACTION_1
    # => exactly three action disagreements.
    z_trans_unit = np.array([
        tL + 0.02,
        tL - 0.02,
        tH + 0.02,
        tH + 0.05,
        3.01,
    ])

    y_dummy = np.array([0, 0, 1, 1, 1])

    metrics, detail = evaluate_decision_invariance(
        y_true=y_dummy,
        z_plain=z_plain_unit,
        z_transformed=z_trans_unit,
        near_boundary_cutoff=0.20,
        transformation_type="UNIT_TEST",
        configuration_id="deterministic_policy_crossing",
    )

    assert metrics["operational_action_disagreement_count"] == 3
    assert np.isclose(
        metrics["operational_action_disagreement_rate"],
        3.0 / 5.0,
    )

    # Any actual operational boundary crossing must have error at least as
    # large as the reference distance to the nearest operational boundary.
    violated = detail["operational_action_disagreement"].to_numpy()
    assert np.all(
        detail.loc[violated, "transformation_error"].to_numpy()
        + 1e-12
        >= detail.loc[violated, "operational_margin"].to_numpy()
    )

    # Identical inputs must yield zero disagreement.
    metrics_same, _ = evaluate_decision_invariance(
        y_true=y_dummy,
        z_plain=z_plain_unit,
        z_transformed=z_plain_unit.copy(),
        near_boundary_cutoff=0.20,
        transformation_type="UNIT_TEST",
        configuration_id="identity",
    )
    assert metrics_same["operational_action_disagreement_count"] == 0
    assert metrics_same["classification_disagreement_count"] == 0

    return {
        "passed": True,
        "tests": [
            "three expected operational boundary crossings",
            "crossing implies error >= operational margin",
            "identity transformation has zero disagreement",
        ],
    }

unit_test_record = run_decision_invariance_unit_tests()

(RUN_DIR / "decision_invariance_unit_tests.json").write_text(
    json.dumps(unit_test_record, indent=2),
    encoding="utf-8",
)

print(unit_test_record)


## 15. Paired bootstrap uncertainty and H3 AUC-difference bootstrap


In [ ]:
def paired_bootstrap_evidence(detail, y_true, reps=2000, seed=42, confidence=0.95):
    rng = np.random.default_rng(seed)
    n = len(detail)
    y_true = np.asarray(y_true)
    action_rates, class_rates, maes, p95_errors, acc_diffs, delta_aucs = [], [], [], [], [], []

    z_plain_all = detail["z_plain"].to_numpy()
    z_trans_all = detail["z_transformed"].to_numpy()
    action_dis_all = detail["operational_action_disagreement"].astype(bool).to_numpy()
    error_all = detail["transformation_error"].to_numpy()
    ratio_all = detail["error_to_margin_ratio"].to_numpy()

    h3_estimable = np.unique(action_dis_all.astype(int)).size == 2

    for _ in range(reps):
        idx = rng.integers(0, n, n)
        zp, zt, yt = z_plain_all[idx], z_trans_all[idx], y_true[idx]
        cp = (zp >= CLASSIFICATION_THRESHOLD).astype(int)
        ct = (zt >= CLASSIFICATION_THRESHOLD).astype(int)
        ap, at = apply_operational_policy(zp), apply_operational_policy(zt)
        action_rates.append(float(np.mean(ap != at)))
        class_rates.append(float(np.mean(cp != ct)))
        abs_err = np.abs(zt - zp)
        maes.append(float(np.mean(abs_err)))
        p95_errors.append(float(np.quantile(abs_err, 0.95)))
        acc_diffs.append(float(abs(accuracy_score(yt, cp) - accuracy_score(yt, ct))))

        if h3_estimable:
            labels = action_dis_all[idx].astype(int)
            if np.unique(labels).size == 2:
                auc_error = roc_auc_score(labels, error_all[idx])
                auc_ratio = roc_auc_score(labels, ratio_all[idx])
                delta_aucs.append(float(auc_ratio - auc_error))

    return {
        "bootstrap_reps_requested": int(reps),
        "bootstrap_seed": int(seed),
        "confidence_level": float(confidence),
        "operational_action_disagreement_ci": percentile_ci(action_rates, confidence),
        "classification_disagreement_ci": percentile_ci(class_rates, confidence),
        "output_mae_ci": percentile_ci(maes, confidence),
        "output_p95_abs_error_ci": percentile_ci(p95_errors, confidence),
        "absolute_binary_accuracy_difference_ci": percentile_ci(acc_diffs, confidence),
        "H3_status": "ESTIMABLE" if h3_estimable else "NOT_ESTIMABLE_ONE_CLASS_DISAGREEMENT_LABEL",
        "H3_delta_auc_bootstrap_reps_valid": int(len(delta_aucs)),
        "H3_delta_auc_ci": percentile_ci(delta_aucs, confidence) if delta_aucs else (None, None),
        "H3_delta_auc_mean": float(np.mean(delta_aucs)) if delta_aucs else None,
    }


## 16. Method-development sensitivity analysis on calibration data only



The simulated stress test is executed on the **calibration split**, not the locked test.

This allows us to verify the behavior of the metrics and plots without consuming the confirmatory test set.

The same deterministic standard-normal direction is scaled by each pre-specified \(\sigma\) to make the stress levels comparable.


**Execution rule (v0.8):** these simulated calibration diagnostics run only in `PILOT` mode. They are not recomputed in `CONFIRMATORY` mode, preventing method-development work from being mixed with frozen confirmatory execution.


In [ ]:
if STUDY_MODE == "PILOT":

    rng = np.random.default_rng(SEED)
    base_noise_direction = rng.normal(0.0, 1.0, size=len(z_cal))

    simulated_metrics = []
    simulated_details = []

    for sigma in SIMULATED_SIGMAS:
        z_sim = z_cal + sigma * base_noise_direction

        metrics, detail = evaluate_decision_invariance(
            y_true=y_cal,
            z_plain=z_cal,
            z_transformed=z_sim,
            near_boundary_cutoff=NEAR_BOUNDARY_CUTOFF,
            transformation_type="SIMULATED_METHOD_VALIDATION",
            configuration_id=f"sigma_{sigma:g}",
        )

        bootstrap = paired_bootstrap_evidence(
            detail=detail,
            y_true=y_cal,
            reps=BOOTSTRAP_REPS,
            seed=SEED + int(round(sigma * 10000)),
            confidence=CI_LEVEL,
        )

        row = {
            **metrics,
            "simulated_sigma": float(sigma),
            "bootstrap_operational_ci_low": bootstrap[
                "operational_action_disagreement_ci"
            ][0],
            "bootstrap_operational_ci_high": bootstrap[
                "operational_action_disagreement_ci"
            ][1],
            "H3_delta_auc_bootstrap_mean": bootstrap[
                "H3_delta_auc_mean"
            ],
            "H3_delta_auc_bootstrap_ci_low": bootstrap[
                "H3_delta_auc_ci"
            ][0],
            "H3_delta_auc_bootstrap_ci_high": bootstrap[
                "H3_delta_auc_ci"
            ][1],
            "H3_delta_auc_bootstrap_reps_valid": bootstrap[
                "H3_delta_auc_bootstrap_reps_valid"
            ],
        }

        detail["simulated_sigma"] = float(sigma)

        simulated_metrics.append(row)
        simulated_details.append(detail)

    sim_metrics_df = pd.DataFrame(simulated_metrics)
    sim_detail_df = pd.concat(simulated_details, ignore_index=True)

    sim_metrics_df.to_csv(
        ROOT / "results" / f"calibration_simulated_sensitivity_metrics_{RUN_ID}.csv",
        index=False,
    )
    sim_detail_df.to_csv(
        ROOT / "results" / f"calibration_simulated_sensitivity_detail_{RUN_ID}.csv",
        index=False,
    )

    display(sim_metrics_df)
else:
    print('CONFIRMATORY: simulated method-development sensitivity analysis skipped.')


## 17. Method-development figures


In [ ]:
if STUDY_MODE == "PILOT":

    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.plot(
        sim_metrics_df["simulated_sigma"],
        sim_metrics_df["classification_disagreement_rate"],
        marker="o",
        label="Classification disagreement",
    )

    ax.plot(
        sim_metrics_df["simulated_sigma"],
        sim_metrics_df["operational_action_disagreement_rate"],
        marker="o",
        label="Operational-action disagreement",
    )

    ax.set_xlabel("Simulated noise sigma")
    ax.set_ylabel("Disagreement rate")
    ax.set_title(
        "Calibration-only method validation: label vs operational disagreement"
    )
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    fig.savefig(
        ROOT / "figures" / f"calibration_disagreement_sensitivity_{RUN_ID}.png",
        dpi=300,
        bbox_inches="tight",
    )
    fig.savefig(
        ROOT / "figures" / f"calibration_disagreement_sensitivity_{RUN_ID}.svg",
        bbox_inches="tight",
    )

    plt.show()
else:
    print('CONFIRMATORY: simulated method-development figure skipped.')


In [ ]:
if STUDY_MODE == "PILOT":

    selected_sigma = max(SIMULATED_SIGMAS)

    selected = sim_detail_df[
        sim_detail_df["simulated_sigma"] == selected_sigma
    ].copy()

    fig, ax = plt.subplots(figsize=(8, 6))

    preserved = ~selected["operational_action_disagreement"]
    changed = selected["operational_action_disagreement"]

    ax.scatter(
        selected.loc[preserved, "operational_margin"],
        selected.loc[preserved, "transformation_error"],
        alpha=0.65,
        label="Operational action preserved",
    )

    ax.scatter(
        selected.loc[changed, "operational_margin"],
        selected.loc[changed, "transformation_error"],
        marker="x",
        s=80,
        linewidths=2,
        label="Operational action changed",
    )

    max_value = max(
        float(selected["operational_margin"].max()),
        float(selected["transformation_error"].max()),
    )

    ax.plot(
        [0, max_value],
        [0, max_value],
        linestyle="--",
        label="error = operational margin",
    )

    ax.set_xlabel("Operational decision margin")
    ax.set_ylabel("Transformation error")
    ax.set_title(
        f"Calibration-only diagnostic — simulated sigma={selected_sigma:g}"
    )
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    fig.savefig(
        ROOT / "figures" / f"calibration_margin_vs_error_{RUN_ID}.png",
        dpi=300,
        bbox_inches="tight",
    )
    fig.savefig(
        ROOT / "figures" / f"calibration_margin_vs_error_{RUN_ID}.svg",
        bbox_inches="tight",
    )

    plt.show()
else:
    print('CONFIRMATORY: simulated diagnostic figure skipped.')


In [ ]:
def _parse_scalar(text):
    text = str(text).strip()
    try:
        if re.fullmatch(r"[-+]?\d+", text):
            return int(text)
        return float(text)
    except ValueError:
        return text

def parse_he_output(stdout, sample_ids, repeats):
    ids = [int(v) for v in sample_ids]
    if len(ids) != len(set(ids)) or not ids or repeats < 1:
        raise ValueError("Expected unique sample IDs and a positive repeat count.")
    rows, timing, runtime_parameters = [], {}, {}
    for line in stdout.splitlines():
        if line.startswith("TIMING_"):
            key, value = line.split(",", 1)
            timing[key] = float(value)
        elif line.startswith("PARAM_"):
            key, value = line.split(",", 1)
            runtime_parameters[key] = _parse_scalar(value)
        elif line.startswith("HE_RESULT,"):
            fields = line.split(",")
            if len(fields) != 7:
                raise ValueError("Malformed HE_RESULT record.")
            rows.append([int(fields[1]), int(fields[2]), *map(float, fields[3:])])

    frame = pd.DataFrame(rows, columns=[
        "repeat", "sample_id", "decrypted_score", "encrypt_ms", "he_eval_ms", "decrypt_ms"
    ])
    expected = {(r, sid) for r in range(repeats) for sid in ids}
    actual = list(zip(frame["repeat"], frame["sample_id"]))
    if len(actual) != len(expected) or set(actual) != expected:
        raise ValueError("Missing, duplicated, or unexpected sample/repetition in HE output.")
    if not np.isfinite(frame.select_dtypes(include="number").to_numpy()).all():
        raise ValueError("Non-finite score or timing in HE output.")
    if (frame[["encrypt_ms", "he_eval_ms", "decrypt_ms"]] < 0).any().any():
        raise ValueError("Negative timing in HE output.")
    if not all(np.isfinite(v) and v >= 0 for v in timing.values()):
        raise ValueError("Invalid setup timing.")
    required_timing = ["TIMING_CONTEXT_SETUP_MS", "TIMING_KEYGEN_MS", "TIMING_CONTEXT_CONFIGURE_MS"]
    if any(key not in timing for key in required_timing):
        raise ValueError("Incomplete setup timing.")
    return frame, timing, runtime_parameters

def ordered_scores(frame, sample_ids, repeat=0):
    return frame.loc[frame["repeat"] == repeat].set_index("sample_id").loc[
        [int(v) for v in sample_ids], "decrypted_score"
    ].to_numpy(dtype=float)

def repeated_decision_evidence(frame, sample_ids, plain, truth, configuration_id, transformation_type):
    metrics_rows, detail_rows = [], []
    for repeat in sorted(frame["repeat"].unique()):
        scores = ordered_scores(frame, sample_ids, int(repeat))
        metrics, detail = evaluate_decision_invariance(
            truth, plain, scores, NEAR_BOUNDARY_CUTOFF, transformation_type, configuration_id
        )
        metrics["repeat"] = int(repeat)
        detail["repeat"] = int(repeat)
        detail["sample_id"] = [int(v) for v in sample_ids]
        metrics_rows.append(metrics)
        detail_rows.append(detail)
    detail = pd.concat(detail_rows, ignore_index=True)
    grouped = detail.groupby("sample_id")["operational_action_disagreement"]
    score_pivot = frame.pivot(index="sample_id", columns="repeat", values="decrypted_score").loc[[int(v) for v in sample_ids]]
    score_spread = score_pivot.max(axis=1) - score_pivot.min(axis=1)
    action_matrix = np.column_stack([apply_operational_policy(score_pivot[col].to_numpy(float)) for col in score_pivot.columns])
    repeat_action_instability = np.array([len(set(row.tolist())) > 1 for row in action_matrix], dtype=bool)
    summary = {
        "primary_repeat": 0,
        "any_repeat_disagreement_rate": float(grouped.any().mean()),
        "all_repeats_disagreement_rate": float(grouped.all().mean()),
        "max_per_repeat_disagreement_rate": float(max(v["operational_action_disagreement_rate"] for v in metrics_rows)),
        "repeat_action_instability_count": int(repeat_action_instability.sum()),
        "repeat_action_instability_rate": float(repeat_action_instability.mean()),
        "max_score_spread_across_repeats": float(score_spread.max()),
        "p95_score_spread_across_repeats": float(np.quantile(score_spread, 0.95)),
        "uncertainty_unit": "sample, never sample-times-repeat",
    }
    return pd.DataFrame(metrics_rows), detail, summary

def execute_he_binary(executable, features, sample_ids, repeats, destination, runtime_lib_dir=None):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    features = np.asarray(features, dtype=np.float32)
    ids = np.asarray(sample_ids)
    if features.ndim != 2 or len(features) != len(ids) or not np.isfinite(features).all():
        raise ValueError("Invalid feature matrix/sample IDs.")
    if len(set(map(int, ids))) != len(ids):
        raise ValueError("Sample IDs are not unique.")

    payload = "\n".join(
        str(int(sid)) + " " + " ".join(f"{float(v):.9e}" for v in row)
        for sid, row in zip(ids, features)
    ) + "\n"
    (destination / "inputs.txt").write_text(payload, encoding="utf-8")
    executable = Path(executable).resolve()
    if not os.access(executable, os.X_OK):
        executable.chmod(executable.stat().st_mode | 0o100)

    env = {**os.environ, "OMP_NUM_THREADS": "2"}
    if runtime_lib_dir is not None:
        frozen_lib_dir = str(Path(runtime_lib_dir).resolve())
        prior_ld_path = env.get("LD_LIBRARY_PATH", "")
        env["LD_LIBRARY_PATH"] = frozen_lib_dir if not prior_ld_path else frozen_lib_dir + os.pathsep + prior_ld_path

    t0 = time.perf_counter()
    proc = subprocess.run(
        [str(executable)], input=payload, capture_output=True, text=True,
        env=env, timeout=HE_EXECUTION_TIMEOUT_SECONDS,
    )
    elapsed = time.perf_counter() - t0
    (destination / "stdout.log").write_text(proc.stdout, encoding="utf-8")
    (destination / "stderr.log").write_text(proc.stderr, encoding="utf-8")
    if proc.returncode:
        raise RuntimeError(f"HE execution failed ({proc.returncode}); inspect {destination / 'stderr.log'}")

    frame, timing, runtime_parameters = parse_he_output(proc.stdout, ids, repeats)
    frame.to_csv(destination / "scores.csv", index=False)
    execution = {
        "command": [str(executable)], "return_code": proc.returncode,
        "wall_seconds": elapsed, "sample_ids": [int(v) for v in ids],
        "repeats": repeats, "executable_sha256": sha256_file(executable),
        "input_sha256": sha256_file(destination / "inputs.txt"),
        "stdout_sha256": sha256_file(destination / "stdout.log"),
        "stderr_sha256": sha256_file(destination / "stderr.log"),
        "scores_sha256": sha256_file(destination / "scores.csv"),
        "timing": timing, "runtime_parameters": runtime_parameters,
    }
    return frame, execution

def run_integrity_self_tests():
    fixture = (
        "TIMING_CONTEXT_SETUP_MS,0\nTIMING_KEYGEN_MS,0\nTIMING_CONTEXT_CONFIGURE_MS,0\n"
        "PARAM_RING_DIMENSION,16384\n"
        "HE_RESULT,0,11,-2,0,0,0\nHE_RESULT,0,29,2,0,0,0\n"
    )
    frame, _, params = parse_he_output(fixture, [11, 29], 1)
    assert params["PARAM_RING_DIMENSION"] == 16384
    np.testing.assert_array_equal(ordered_scores(frame, [29, 11]), [2, -2])
    rejected = 0
    for malformed in [
        fixture + "HE_RESULT,0,11,-2,0,0,0\n",
        fixture.replace("HE_RESULT,0,29,2,0,0,0\n", ""),
        fixture.replace("HE_RESULT,0,29", "HE_RESULT,0,99"),
        fixture.replace("HE_RESULT,0,29,2", "HE_RESULT,0,29,nan"),
    ]:
        try:
            parse_he_output(malformed, [11, 29], 1)
        except ValueError:
            rejected += 1
        else:
            raise AssertionError("Invalid execution output was accepted.")
    try:
        evaluate_decision_invariance([0], [0], [np.inf], .1, "UNIT_TEST", "fixture")
    except ValueError:
        rejected += 1
    else:
        raise AssertionError("Non-finite score was accepted.")
    record = {
        "type": "UNIT_TEST", "passed": True,
        "invalid_inputs_rejected": rejected,
        "sample_order_verified": True,
        "runtime_parameter_parser_verified": True,
        "is_he_execution_evidence": False,
    }
    (RUN_DIR / "integrity_self_tests.json").write_text(json.dumps(record, indent=2))
    return record

print(run_integrity_self_tests())


## 18. Automated real HEIR + CKKS calibration and boundary-stress study

Both pre-specified configurations are executed on the complete calibration split and on the deterministic calibration-derived boundary stress set. The stress execution uses the exact same generated executable and runtime stack as the calibration execution; only the input set changes.


In [ ]:
if STUDY_MODE == "PILOT":
    if CKKS_PILOT_SAMPLE_LIMIT is not None and (
        isinstance(CKKS_PILOT_SAMPLE_LIMIT, bool)
        or not isinstance(CKKS_PILOT_SAMPLE_LIMIT, int)
        or CKKS_PILOT_SAMPLE_LIMIT < 1
    ):
        raise ValueError("CKKS_PILOT_SAMPLE_LIMIT must be None or a positive integer.")

    CKKS_PILOT_N = len(X_cal) if CKKS_PILOT_SAMPLE_LIMIT is None else min(CKKS_PILOT_SAMPLE_LIMIT, len(X_cal))
    w_ckks = w_export_f32.copy()
    b_ckks = np.float32(b_export_f32)
    X_cal_f32 = X_cal.to_numpy(dtype=np.float32)
    z_cal_export_f32 = float32_export_score(X_cal)

    def mlir_float32_literal(value):
        v = float(np.float32(value))
        if not np.isfinite(v):
            raise ValueError("Non-finite model coefficient cannot be exported.")
        return f"{v:.9e}"

    def build_affine_score_mlir(weights, bias):
        weights = np.asarray(weights, dtype=np.float32)
        n = len(weights)
        dense = ", ".join(mlir_float32_literal(v) for v in weights)
        return f'''module {{
  func.func @affine_score(
      %arg0: tensor<{n}xf32> {{secret.secret}}
  ) -> f32 {{
    %weights = arith.constant dense<[{dense}]> : tensor<{n}xf32>
    %bias = arith.constant {mlir_float32_literal(bias)} : f32
    %zero = arith.constant 0.000000000e+00 : f32
    %sum = affine.for %i = 0 to {n} iter_args(%acc = %zero) -> (f32) {{
      %x = tensor.extract %arg0[%i] : tensor<{n}xf32>
      %w = tensor.extract %weights[%i] : tensor<{n}xf32>
      %prod = arith.mulf %x, %w : f32
      %next = arith.addf %acc, %prod : f32
      affine.yield %next : f32
    }}
    %result = arith.addf %sum, %bias : f32
    return %result : f32
  }}
}}
'''

    def translate_to_file(heir_translate, mlir_path, args, output_path, log_path):
        proc = subprocess.run([heir_translate, *args, str(mlir_path)], text=True, capture_output=True)
        log_path.write_text(
            "COMMAND:\n" + " ".join([heir_translate, *args, str(mlir_path)])
            + "\n\nSTDOUT:\n" + (proc.stdout or "")
            + "\n\nSTDERR:\n" + (proc.stderr or ""),
            encoding="utf-8",
        )
        if proc.returncode != 0:
            raise RuntimeError(f"HEIR translation failed. See {log_path}\n{((proc.stderr or proc.stdout or '')[-6000:])}")
        output_path.write_text(proc.stdout, encoding="utf-8")

    def find_openfhe_library_dirs(config_dir):
        roots = [Path("/usr/local/lib"), Path("/usr/local/lib64"), Path("/usr/lib"), Path("/usr/lib64"), Path("/usr/lib/x86_64-linux-gnu")]
        dirs, files = set(), []
        for root in roots:
            if not root.exists():
                continue
            for pattern in ["libOPENFHE*.so", "libOPENFHE*.so.*", "libOPENFHE*.a"]:
                for lib in root.rglob(pattern):
                    dirs.add(lib.parent)
                    files.append(str(lib))
        result = sorted(dirs, key=str)
        (config_dir / "openfhe_library_discovery.json").write_text(
            json.dumps({"library_dirs": [str(p) for p in result], "library_files": sorted(set(files))}, indent=2),
            encoding="utf-8",
        )
        if not result:
            raise RuntimeError("No installed OpenFHE library directories were found.")
        return result

    def run_cmake_stage(config_dir, cmd, log_name):
        proc = subprocess.run([str(v) for v in cmd], text=True, capture_output=True)
        combined = (
            "COMMAND:\n" + " ".join(map(str, cmd))
            + "\n\nSTDOUT:\n" + (proc.stdout or "")
            + "\n\nSTDERR:\n" + (proc.stderr or "")
        )
        log_path = config_dir / log_name
        log_path.write_text(combined, encoding="utf-8")
        if proc.returncode != 0:
            raise RuntimeError(
                f"CMake stage failed ({log_name}). Full log: {log_path}.\n"
                f"Last compiler/configuration output:\n{combined[-8000:]}"
            )
        return proc

    def extract_generated_parameters(lowered_text, cpp_text, runtime_parameters, requested_spec):
        def int_match(pattern, text):
            m = re.search(pattern, text)
            return int(m.group(1)) if m else None
        def str_match(pattern, text):
            m = re.search(pattern, text)
            return m.group(1) if m else None

        explicit_security = str_match(r"SetSecurityLevel\((HEStd_[A-Za-z0-9_]+)\)", cpp_text)
        explicit_ring = int_match(r"SetRingDim\((\d+)\)", cpp_text)
        actual_slots = int_match(r"scheme\.actual_slot_count\s*=\s*(\d+)", lowered_text)
        requested_slots = int_match(r"scheme\.requested_slot_count\s*=\s*(\d+)", lowered_text)
        ring_runtime = runtime_parameters.get("PARAM_RING_DIMENSION")

        params = {
            "security_level": explicit_security,
            "security_level_provenance": (
                "explicit SetSecurityLevel in generated C++" if explicit_security
                else "not explicitly set by generated C++; OpenFHE default applies; no independent security-level claim made"
            ),
            "ring_dimension": int(ring_runtime) if isinstance(ring_runtime, (int, float)) else explicit_ring,
            "ring_dimension_provenance": "runtime CryptoContext::GetRingDimension" if ring_runtime is not None else "generated SetRingDim",
            "multiplicative_depth": int_match(r"SetMultiplicativeDepth\((\d+)\)", cpp_text),
            "first_mod_size": int_match(r"SetFirstModSize\((\d+)\)", cpp_text),
            "scaling_mod_size": int_match(r"SetScalingModSize\((\d+)\)", cpp_text),
            "key_switch_technique": str_match(r"SetKeySwitchTechnique\(([A-Za-z0-9_]+)\)", cpp_text),
            "actual_slot_count": actual_slots,
            "requested_slot_count_from_ir": requested_slots,
            "requested_configuration": requested_spec,
        }
        for key in ["ring_dimension", "multiplicative_depth", "first_mod_size", "scaling_mod_size"]:
            if params[key] is None:
                raise RuntimeError(f"Could not normalize required generated/runtime CKKS parameter: {key}")
        ring = params["ring_dimension"]
        if ring < 2 or ring & (ring - 1):
            raise RuntimeError(f"Unexpected non-power-of-two ring dimension: {ring}")
        if params["first_mod_size"] != int(requested_spec["first_mod_bits"]):
            raise RuntimeError(
                f"Generated first modulus {params['first_mod_size']} differs from requested "
                f"{requested_spec['first_mod_bits']}."
            )
        if params["scaling_mod_size"] != int(requested_spec["scaling_mod_bits"]):
            raise RuntimeError(
                f"Generated scaling modulus {params['scaling_mod_size']} differs from requested "
                f"{requested_spec['scaling_mod_bits']}."
            )
        if requested_slots is not None and requested_slots != int(requested_spec["min_slot_count"]):
            raise RuntimeError("Lowered IR requested-slot-count differs from the pre-specified configuration.")
        if actual_slots is not None and actual_slots < int(requested_spec["min_slot_count"]):
            raise RuntimeError("Lowered IR actual slot count is smaller than the requested minimum.")
        return params

    def run_ckks_configuration(spec):
        configuration_id = checked_id(spec["configuration_id"])
        if spec["min_slot_count"] < len(X.columns):
            raise ValueError(f"{configuration_id}: min_slot_count must cover all features.")

        config_dir = ROOT / "artifacts" / configuration_id
        if config_dir.exists():
            raise FileExistsError(f"Configuration artifact directory already exists: {config_dir}")
        config_dir.mkdir(parents=True)

        input_mlir = config_dir / "affine_score_input.mlir"
        openfhe_mlir = config_dir / "affine_score_openfhe.mlir"
        cpp_path = config_dir / "affine_score.cpp"
        header_path = config_dir / "affine_score.h"
        lowering_log = config_dir / "affine_score_lowering.log"
        input_mlir.write_text(build_affine_score_mlir(w_ckks, b_ckks), encoding="utf-8")

        heir_opt = shutil.which("heir-opt")
        heir_translate = shutil.which("heir-translate")
        if not heir_opt or not heir_translate:
            raise RuntimeError("heir-opt/heir-translate unavailable.")

        mlir_to_ckks_opts = [
            f"min-slot-count={int(spec['min_slot_count'])}",
            f"first-mod-bits={int(spec['first_mod_bits'])}",
            f"scaling-mod-bits={int(spec['scaling_mod_bits'])}",
        ]
        lower_cmd = [
            heir_opt,
            "--annotate-module=backend=openfhe scheme=ckks",
            "--mlir-to-ckks=" + " ".join(mlir_to_ckks_opts),
            "--scheme-to-openfhe=entry-function=affine_score",
            str(input_mlir),
        ]
        lower_proc = subprocess.run(lower_cmd, text=True, capture_output=True)
        lowering_log.write_text(
            "COMMAND:\n" + " ".join(lower_cmd)
            + "\n\nSTDOUT:\n" + (lower_proc.stdout or "")
            + "\n\nSTDERR:\n" + (lower_proc.stderr or ""), encoding="utf-8"
        )
        if lower_proc.returncode != 0:
            raise RuntimeError(
                f"{configuration_id}: HEIR CKKS lowering failed. See {lowering_log}\n"
                + ((lower_proc.stderr or lower_proc.stdout or "")[-8000:])
            )
        openfhe_mlir.write_text(lower_proc.stdout, encoding="utf-8")

        translate_to_file(
            heir_translate, openfhe_mlir,
            ["--emit-openfhe-pke", "--openfhe-include-type=install-relative"],
            cpp_path, config_dir / "emit_openfhe_cpp.log",
        )
        translate_to_file(
            heir_translate, openfhe_mlir,
            ["--emit-openfhe-pke-header", "--openfhe-include-type=install-relative"],
            header_path, config_dir / "emit_openfhe_header.log",
        )

        optional_emitters = {}
        for flag, filename, label in [
            ("--emit-metadata", "affine_score_metadata.json", "metadata"),
            ("--emit-function-info", "affine_score_function_info.txt", "function_info"),
        ]:
            proc = subprocess.run([heir_translate, flag, str(openfhe_mlir)], text=True, capture_output=True)
            (config_dir / f"emit_{label}.log").write_text(
                "STDOUT:\n" + (proc.stdout or "") + "\n\nSTDERR:\n" + (proc.stderr or ""), encoding="utf-8"
            )
            optional_emitters[label] = {"return_code": int(proc.returncode), "available": bool(proc.returncode == 0)}
            if proc.returncode == 0:
                (config_dir / filename).write_text(proc.stdout, encoding="utf-8")
        (config_dir / "optional_heir_emitters.json").write_text(json.dumps(optional_emitters, indent=2), encoding="utf-8")

        lowered_text = openfhe_mlir.read_text(encoding="utf-8")
        cpp_text = cpp_path.read_text(encoding="utf-8")
        header_text = header_path.read_text(encoding="utf-8")
        expected_symbols = [
            "affine_score(", "affine_score__encrypt__arg0(", "affine_score__decrypt__result0(",
            "affine_score__generate_crypto_context(", "affine_score__configure_crypto_context(",
        ]
        missing = [s for s in expected_symbols if s not in header_text]
        if missing:
            raise RuntimeError(f"{configuration_id}: generated HEIR client API is missing: {missing}")
        (config_dir / "generated_header_snapshot.txt").write_text(header_text, encoding="utf-8")

        OPENFHE_USER_CMAKE = OPENFHE_SRC / "CMakeLists.User.txt"
        if not OPENFHE_USER_CMAKE.is_file():
            raise RuntimeError("Pinned OpenFHE source does not contain CMakeLists.User.txt.")
        openfhe_user_template = OPENFHE_USER_CMAKE.read_text(encoding="utf-8")
        openfhe_cmake_sha = sha256_file(OPENFHE_USER_CMAKE)

        probe_dir = config_dir / "cmake_codegen_probe"
        probe_build = probe_dir / "build"
        probe_dir.mkdir(parents=True)
        probe_cmake = openfhe_user_template + (
            "\n\n# HEIR compile-only probe\n"
            f'add_library(affine_score_codegen OBJECT "{cpp_path.as_posix()}")\n'
            "set_target_properties(affine_score_codegen PROPERTIES POSITION_INDEPENDENT_CODE ON)\n"
            "target_compile_options(affine_score_codegen PRIVATE -Wno-error=sign-compare)\n"
        )
        (probe_dir / "CMakeLists.txt").write_text(probe_cmake, encoding="utf-8")
        (config_dir / "CMakeLists.codegen_probe.txt").write_text(probe_cmake, encoding="utf-8")
        run_cmake_stage(config_dir, ["cmake", "-S", probe_dir, "-B", probe_build, "-DCMAKE_BUILD_TYPE=Release"], "cmake_codegen_configure.log")
        run_cmake_stage(config_dir, ["cmake", "--build", probe_build, "--target", "affine_score_codegen", "--", "-j2"], "cmake_codegen_build.log")
        # Build trees are transient and can be very large; logs and exact CMake text are preserved.
        shutil.rmtree(probe_dir, ignore_errors=True)

        pilot_X = X_cal_f32[:CKKS_PILOT_N]
        pilot_plain = z_cal[:CKKS_PILOT_N]
        pilot_export_f32 = z_cal_export_f32[:CKKS_PILOT_N]
        harness_cpp = config_dir / "affine_score_pilot_main.cpp"
        harness_source = f'''#include <chrono>
#include <cstdlib>
#include <iomanip>
#include <iostream>
#include <vector>
#include "affine_score.h"
using Clock = std::chrono::high_resolution_clock;
double elapsed_ms(const Clock::time_point& start, const Clock::time_point& end) {{
  return std::chrono::duration<double, std::milli>(end - start).count();
}}
int main() {{
  auto setup_start = Clock::now();
  auto cryptoContext = affine_score__generate_crypto_context();
  auto setup_end = Clock::now();
  std::cout << std::setprecision(17);
  std::cout << "PARAM_RING_DIMENSION," << cryptoContext->GetRingDimension() << std::endl;
  auto keygen_start = Clock::now();
  KeyPair<DCRTPoly> keyPair = cryptoContext->KeyGen();
  auto keygen_end = Clock::now();
  auto configure_start = Clock::now();
  cryptoContext = affine_score__configure_crypto_context(cryptoContext, keyPair.secretKey);
  auto configure_end = Clock::now();
  std::vector<std::vector<float>> samples;
  std::vector<long long> sampleIds;
  long long sampleId;
  while (std::cin >> sampleId) {{
    std::vector<float> row({len(w_ckks)});
    for (auto& value : row) {{
      if (!(std::cin >> value)) {{ std::cerr << "Incomplete input row"; return 2; }}
    }}
    sampleIds.push_back(sampleId);
    samples.push_back(row);
  }}
  if (!std::cin.eof() || samples.empty()) {{ std::cerr << "Invalid or empty input"; return 3; }}
  std::cout << "TIMING_CONTEXT_SETUP_MS," << elapsed_ms(setup_start, setup_end) << std::endl;
  std::cout << "TIMING_KEYGEN_MS," << elapsed_ms(keygen_start, keygen_end) << std::endl;
  std::cout << "TIMING_CONTEXT_CONFIGURE_MS," << elapsed_ms(configure_start, configure_end) << std::endl;
  for (int repeat = 0; repeat < {CKKS_RUNTIME_REPEATS}; ++repeat) {{
    for (std::size_t i = 0; i < samples.size(); ++i) {{
      auto enc_start = Clock::now();
      auto encrypted = affine_score__encrypt__arg0(cryptoContext, samples[i], keyPair.publicKey);
      auto enc_end = Clock::now();
      auto eval_start = Clock::now();
      auto encryptedResult = affine_score(cryptoContext, encrypted);
      auto eval_end = Clock::now();
      auto dec_start = Clock::now();
      auto decrypted = affine_score__decrypt__result0(cryptoContext, encryptedResult, keyPair.secretKey);
      auto dec_end = Clock::now();
      std::cout << "HE_RESULT," << repeat << "," << sampleIds[i] << "," << decrypted << ","
                << elapsed_ms(enc_start, enc_end) << "," << elapsed_ms(eval_start, eval_end) << ","
                << elapsed_ms(dec_start, dec_end) << std::endl;
    }}
  }}
  return 0;
}}
'''
        harness_cpp.write_text(harness_source, encoding="utf-8")

        library_dirs = find_openfhe_library_dirs(config_dir)
        runtime_dir = config_dir / "cmake_runtime"
        runtime_build = runtime_dir / "build"
        runtime_dir.mkdir(parents=True)
        runtime_cmake = openfhe_user_template + (
            "\n\n# HEIR calibration executable\n"
            "add_executable(affine_score_pilot\n"
            f'    "{cpp_path.as_posix()}"\n'
            f'    "{harness_cpp.as_posix()}"\n'
            ")\n"
            f'target_include_directories(affine_score_pilot PRIVATE "{config_dir.as_posix()}")\n'
            "target_compile_options(affine_score_pilot PRIVATE -Wno-error=sign-compare)\n"
        )
        (runtime_dir / "CMakeLists.txt").write_text(runtime_cmake, encoding="utf-8")
        (config_dir / "CMakeLists.runtime.txt").write_text(runtime_cmake, encoding="utf-8")
        run_cmake_stage(config_dir, ["cmake", "-S", runtime_dir, "-B", runtime_build, "-DCMAKE_BUILD_TYPE=Release"], "cmake_runtime_configure.log")
        run_cmake_stage(config_dir, ["cmake", "--build", runtime_build, "--target", "affine_score_pilot", "--", "-j2"], "cmake_runtime_build.log")
        candidates = [p for p in runtime_build.rglob("affine_score_pilot") if p.is_file() and os.access(p, os.X_OK)]
        if not candidates:
            raise RuntimeError(f"{configuration_id}: CMake succeeded but executable was not found.")
        executable = config_dir / "affine_score_pilot"
        shutil.copy2(sorted(candidates, key=lambda p: len(str(p)))[0], executable)
        executable.chmod(executable.stat().st_mode | 0o111)
        shutil.rmtree(runtime_dir, ignore_errors=True)

        ldd_probe = subprocess.run(["ldd", str(executable)], text=True, capture_output=True, check=False)
        (config_dir / "pilot_ldd_before_execution.txt").write_text(
            (ldd_probe.stdout or "") + ("\nSTDERR:\n" + ldd_probe.stderr if ldd_probe.stderr else ""), encoding="utf-8"
        )
        if ldd_probe.returncode != 0 or "not found" in (ldd_probe.stdout or ""):
            raise RuntimeError(f"{configuration_id}: unresolved dynamic dependency before execution.")

        pilot_ids = cal_idx[:CKKS_PILOT_N]
        execution_dir = RUN_DIR / "pilot_executions" / configuration_id
        runtime_detail, execution = execute_he_binary(executable, pilot_X, pilot_ids, CKKS_RUNTIME_REPEATS, execution_dir)
        write_json_exclusive(execution_dir / "execution.json", execution)
        z_he = ordered_scores(runtime_detail, pilot_ids, repeat=0)

        metrics, detail = evaluate_decision_invariance(
            y_cal.iloc[:CKKS_PILOT_N], pilot_plain, z_he, NEAR_BOUNDARY_CUTOFF,
            "HEIR_REAL_HE_PILOT_CALIBRATION", configuration_id,
        )
        bootstrap = paired_bootstrap_evidence(
            detail, np.asarray(y_cal.iloc[:CKKS_PILOT_N]), reps=BOOTSTRAP_REPS,
            seed=SEED + 2000 + sum(ord(ch) for ch in configuration_id), confidence=CI_LEVEL,
        )
        detail["sample_id"] = [int(v) for v in pilot_ids]
        detail["z_plain_float32_export"] = pilot_export_f32
        detail["float32_export_error_vs_float64"] = np.abs(pilot_export_f32 - pilot_plain)
        detail["he_error_vs_float32_export"] = np.abs(z_he - pilot_export_f32)

        score_summary = runtime_detail.groupby("sample_id").agg(
            decrypted_score_median=("decrypted_score", "median"),
            decrypted_score_min=("decrypted_score", "min"),
            decrypted_score_max=("decrypted_score", "max"),
        )
        runtime_summary = {
            "context_setup_ms": execution["timing"]["TIMING_CONTEXT_SETUP_MS"],
            "keygen_ms": execution["timing"]["TIMING_KEYGEN_MS"],
            "context_configure_ms": execution["timing"]["TIMING_CONTEXT_CONFIGURE_MS"],
            "runtime_repeats": CKKS_RUNTIME_REPEATS,
            "pilot_samples": CKKS_PILOT_N,
            "wall_runtime_seconds": float(execution["wall_seconds"]),
            "primary_repeat": 0,
            "max_repeat_score_spread": float((score_summary.decrypted_score_max - score_summary.decrypted_score_min).max()),
            **{f"{col}_median": float(runtime_detail[col].median()) for col in ["encrypt_ms", "he_eval_ms", "decrypt_ms"]},
        }
        metrics.update({
            "pilot_wall_runtime_seconds": runtime_summary["wall_runtime_seconds"],
            "context_setup_ms": runtime_summary["context_setup_ms"],
            "keygen_ms": runtime_summary["keygen_ms"],
            "context_configure_ms": runtime_summary["context_configure_ms"],
            "encrypt_ms_median": runtime_summary["encrypt_ms_median"],
            "he_eval_ms_median": runtime_summary["he_eval_ms_median"],
            "decrypt_ms_median": runtime_summary["decrypt_ms_median"],
            "runtime_repeats": CKKS_RUNTIME_REPEATS,
            "max_repeat_score_spread": runtime_summary["max_repeat_score_spread"],
            "float32_export_mae_vs_float64_plaintext": float(np.mean(np.abs(pilot_export_f32 - pilot_plain))),
            "float32_export_p95_abs_error_vs_float64_plaintext": float(np.quantile(np.abs(pilot_export_f32 - pilot_plain), 0.95)),
            "he_mae_vs_float32_export": float(np.mean(np.abs(z_he - pilot_export_f32))),
            "he_p95_abs_error_vs_float32_export": float(np.quantile(np.abs(z_he - pilot_export_f32), 0.95)),
            "he_p99_abs_error_vs_float32_export": float(np.quantile(np.abs(z_he - pilot_export_f32), 0.99)),
        })

        # Pre-specified sensitivity tables on the calibration split.
        boundary_sensitivity = boundary_cutoff_sensitivity(detail, NEAR_BOUNDARY_CUTOFFS)
        policy_sensitivity = policy_threshold_sensitivity(pilot_plain, z_he, POLICY_SENSITIVITY_PROB_PAIRS)
        boundary_sensitivity.to_csv(config_dir / "calibration_boundary_cutoff_sensitivity.csv", index=False)
        policy_sensitivity.to_csv(config_dir / "calibration_policy_threshold_sensitivity.csv", index=False)

        # Mechanistic calibration-derived boundary stress set. This is diagnostic only.
        stress_execution_dir = RUN_DIR / "boundary_stress_executions" / configuration_id
        stress_runtime, stress_execution = execute_he_binary(
            executable, BOUNDARY_STRESS_X.to_numpy(dtype=np.float32), BOUNDARY_STRESS_IDS,
            CKKS_RUNTIME_REPEATS, stress_execution_dir
        )
        write_json_exclusive(stress_execution_dir / "execution.json", stress_execution)
        stress_he = ordered_scores(stress_runtime, BOUNDARY_STRESS_IDS, repeat=0)
        stress = BOUNDARY_STRESS_META.copy()
        stress["he_score"] = stress_he
        stress["end_to_end_error"] = np.abs(stress["he_score"] - stress["achieved_score_float64"])
        stress["he_incremental_error"] = np.abs(stress["he_score"] - stress["float32_export_score"])
        stress["action_float64"] = apply_operational_policy(stress["achieved_score_float64"].to_numpy(float))
        stress["action_float32_export"] = apply_operational_policy(stress["float32_export_score"].to_numpy(float))
        stress["action_he"] = apply_operational_policy(stress["he_score"].to_numpy(float))
        stress["serialization_crossing"] = stress["action_float64"] != stress["action_float32_export"]
        stress["he_incremental_crossing"] = stress["action_float32_export"] != stress["action_he"]
        stress["end_to_end_crossing"] = stress["action_float64"] != stress["action_he"]
        stress.to_csv(config_dir / "boundary_stress_detail.csv", index=False)
        in_range = stress["within_calibration_marginal_minmax"].astype(bool)
        plausible = stress["plausible_joint_support"].astype(bool)
        stress_summary = {
            "evidence_label": "HEIR_BOUNDARY_STRESS_DIAGNOSTIC",
            "n": int(len(stress)),
            "serialization_crossing_count": int(stress["serialization_crossing"].sum()),
            "he_incremental_crossing_count": int(stress["he_incremental_crossing"].sum()),
            "end_to_end_crossing_count": int(stress["end_to_end_crossing"].sum()),
            "in_calibration_minmax_n": int(in_range.sum()),
            "in_calibration_minmax_he_incremental_crossing_count": int(stress.loc[in_range, "he_incremental_crossing"].sum()),
            "in_calibration_minmax_end_to_end_crossing_count": int(stress.loc[in_range, "end_to_end_crossing"].sum()),
            "plausible_joint_support_n": int(plausible.sum()),
            "plausible_joint_support_he_incremental_crossing_count": int(stress.loc[plausible, "he_incremental_crossing"].sum()),
            "plausible_joint_support_end_to_end_crossing_count": int(stress.loc[plausible, "end_to_end_crossing"].sum()),
            "plausibility_rule": BOUNDARY_STRESS_PLAUSIBILITY,
            "median_he_incremental_error": float(stress["he_incremental_error"].median()),
            "p95_he_incremental_error": float(stress["he_incremental_error"].quantile(0.95)),
            "max_he_incremental_error": float(stress["he_incremental_error"].max()),
            "population_inference_allowed": False,
        }
        (config_dir / "boundary_stress_summary.json").write_text(json.dumps(stress_summary, indent=2), encoding="utf-8")
        stress.groupby(["boundary", "delta"], as_index=False).agg(
            n=("sample_id", "size"),
            serialization_crossings=("serialization_crossing", "sum"),
            he_incremental_crossings=("he_incremental_crossing", "sum"),
            end_to_end_crossings=("end_to_end_crossing", "sum"),
            median_he_incremental_error=("he_incremental_error", "median"),
            max_he_incremental_error=("he_incremental_error", "max"),
        ).to_csv(config_dir / "boundary_stress_by_delta.csv", index=False)
        stress.groupby(["plausible_joint_support"], as_index=False).agg(
            n=("sample_id", "size"),
            serialization_crossings=("serialization_crossing", "sum"),
            he_incremental_crossings=("he_incremental_crossing", "sum"),
            end_to_end_crossings=("end_to_end_crossing", "sum"),
            median_he_incremental_error=("he_incremental_error", "median"),
            p95_he_incremental_error=("he_incremental_error", lambda x: float(np.quantile(x, 0.95))),
        ).to_csv(config_dir / "boundary_stress_by_plausibility.csv", index=False)
        metrics.update({
            "boundary_stress_serialization_crossing_count": stress_summary["serialization_crossing_count"],
            "boundary_stress_he_incremental_crossing_count": stress_summary["he_incremental_crossing_count"],
            "boundary_stress_end_to_end_crossing_count": stress_summary["end_to_end_crossing_count"],
            "boundary_stress_plausible_joint_support_n": stress_summary["plausible_joint_support_n"],
            "boundary_stress_plausible_he_incremental_crossing_count": stress_summary["plausible_joint_support_he_incremental_crossing_count"],
            "boundary_stress_plausible_end_to_end_crossing_count": stress_summary["plausible_joint_support_end_to_end_crossing_count"],
        })

        normalized_parameters = extract_generated_parameters(lowered_text, cpp_text, execution["runtime_parameters"], spec)
        evidence = {
            "normalized_parameters": normalized_parameters,
            "module_attribute_lines": [line.strip() for line in lowered_text.splitlines() if "scheme." in line or "backend.openfhe" in line][:50],
            "openfhe_context_parameter_lines": [
                line.strip() for line in cpp_text.splitlines()
                if any(token in line for token in [
                    "SetSecurityLevel", "SetRingDim", "SetMultiplicativeDepth", "SetScalingModSize",
                    "SetFirstModSize", "SetScalingTechnique", "SetKeySwitchTechnique", "SetBatchSize",
                ])
            ][:100],
            "runtime_parameters": execution["runtime_parameters"],
        }
        (config_dir / "compiler_parameter_evidence.json").write_text(json.dumps(evidence, indent=2), encoding="utf-8")
        (config_dir / "runtime_summary.json").write_text(json.dumps(runtime_summary, indent=2), encoding="utf-8")
        score_summary.to_csv(config_dir / "repeated_score_summary.csv")

        repeat_metrics, repeat_detail, repeat_summary = repeated_decision_evidence(
            runtime_detail, pilot_ids, pilot_plain, y_cal.iloc[:CKKS_PILOT_N],
            configuration_id, "HEIR_REAL_HE_PILOT_CALIBRATION",
        )
        repeat_metrics.to_csv(config_dir / "per_repeat_metrics.csv", index=False)
        repeat_detail.to_csv(config_dir / "all_repetitions_detail.csv", index=False)
        (config_dir / "repeated_decisions.json").write_text(json.dumps(repeat_summary, indent=2), encoding="utf-8")
        pd.DataFrame([metrics]).to_csv(ROOT / "results" / f"{configuration_id}_metrics.csv", index=False)
        detail.to_csv(ROOT / "results" / f"{configuration_id}_detail.csv", index=False)
        (ROOT / "results" / f"{configuration_id}_bootstrap.json").write_text(json.dumps(bootstrap, indent=2), encoding="utf-8")

        generated_artifacts = [
            input_mlir, openfhe_mlir, cpp_path, header_path, harness_cpp,
            config_dir / "CMakeLists.codegen_probe.txt", config_dir / "cmake_codegen_configure.log",
            config_dir / "cmake_codegen_build.log", config_dir / "CMakeLists.runtime.txt",
            config_dir / "cmake_runtime_configure.log", config_dir / "cmake_runtime_build.log",
            config_dir / "pilot_ldd_before_execution.txt", config_dir / "compiler_parameter_evidence.json",
            config_dir / "optional_heir_emitters.json",
            config_dir / "calibration_boundary_cutoff_sensitivity.csv",
            config_dir / "calibration_policy_threshold_sensitivity.csv",
            config_dir / "boundary_stress_detail.csv", config_dir / "boundary_stress_summary.json",
            config_dir / "boundary_stress_by_delta.csv", config_dir / "boundary_stress_by_plausibility.csv",
        ]
        record = {
            "configuration_id": configuration_id,
            "status": "EXECUTED",
            "science_sha256": digest_json(scientific_snapshot()),
            "stage": "CALIBRATION_PILOT",
            "scheme": "CKKS", "backend": "OpenFHE",
            "requested_configuration": spec,
            "normalized_parameters": normalized_parameters,
            "heir_version": environment.get("heir_py_version"),
            "heir_release_commit": HEIR_RELEASE_COMMIT,
            "heir_expected_openfhe_bcr_version": HEIR_OPENFHE_BCR_VERSION,
            "openfhe_git_ref": OPENFHE_GIT_REF,
            "openfhe_commit": resolved_openfhe_sha,
            "openfhe_cmake_template_sha256": openfhe_cmake_sha,
            "input_features": int(len(w_ckks)),
            "input_type": f"tensor<{len(w_ckks)}xf32> secret",
            "heir_pipeline": [
                "annotate-module backend=openfhe scheme=ckks",
                "mlir-to-ckks " + " ".join(mlir_to_ckks_opts),
                "scheme-to-openfhe entry-function=affine_score",
            ],
            "compile_command": lower_cmd,
            "generated_artifacts": [str(p.relative_to(ROOT)) for p in generated_artifacts if p.exists()],
            "pilot_samples": CKKS_PILOT_N,
            "runtime_wall_seconds_total": float(execution["wall_seconds"]),
            "runtime_repeats": CKKS_RUNTIME_REPEATS,
            "runtime_summary_file": str((config_dir / "runtime_summary.json").relative_to(ROOT)),
            "openfhe_library_dirs": [str(p) for p in library_dirs],
            "notes": "Calibration-stage real-HE pilot; not locked-test confirmatory evidence.",
        }
        (config_dir / "configuration_executed.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
        return {"record": record, "metrics": metrics, "detail": detail, "bootstrap": bootstrap}

    PILOT_CONFIGURATION_RESULTS = {}
    for spec in HE_CONFIGURATION_SPECS:
        cid = spec["configuration_id"]
        print("\n=== Running pre-specified HE configuration:", cid, "===")
        PILOT_CONFIGURATION_RESULTS[cid] = run_ckks_configuration(spec)
        print(json.dumps(PILOT_CONFIGURATION_RESULTS[cid]["metrics"], indent=2))

    pilot_metrics_table = pd.DataFrame([v["metrics"] for v in PILOT_CONFIGURATION_RESULTS.values()])
    pilot_metrics_table.to_csv(ROOT / "results" / "pilot_real_he_configuration_summary.csv", index=False)
    pilot_readiness = {
        "full_calibration_used": bool(CKKS_PILOT_SAMPLE_LIMIT is None and CKKS_PILOT_N == len(X_cal)),
        "all_pre_specified_configurations_executed": bool(
            set(PILOT_CONFIGURATION_RESULTS) == {spec["configuration_id"] for spec in HE_CONFIGURATION_SPECS}
        ),
        "any_operational_disagreement_observed_in_calibration": bool(
            pilot_metrics_table["operational_action_disagreement_count"].gt(0).any()
        ),
        "any_boundary_stress_he_incremental_crossing_observed": bool(
            pilot_metrics_table["boundary_stress_he_incremental_crossing_count"].gt(0).any()
        ),
        "any_boundary_stress_end_to_end_crossing_observed": bool(
            pilot_metrics_table["boundary_stress_end_to_end_crossing_count"].gt(0).any()
        ),
        "boundary_stress_interpretation": "mechanistic diagnostic only; not a population incidence estimate",
        "H2_H3_note": (
            "If no operational disagreements occur, H2/H3 remain legitimately not estimable; "
            "the study does not tune parameters post hoc to manufacture disagreements."
        ),
    }
    (ROOT / "results" / "pilot_study_readiness.json").write_text(
        json.dumps(pilot_readiness, indent=2), encoding="utf-8"
    )
    display(pilot_metrics_table)
    print(json.dumps(pilot_readiness, indent=2))
else:
    print("CONFIRMATORY: calibration compilation/pilot skipped; frozen artifacts will be reused.")


### 18.1–18.9 implementation note
The complete build/execute/evaluate path is intentionally kept in one code cell so each pre-specified configuration follows exactly the same implementation and cannot diverge because of manual re-execution of selected sub-cells.


## 19. Pre-specified multi-configuration comparison

The pilot comparison is descriptive calibration evidence. The confirmatory H4 analysis uses the same two frozen configurations and a paired bootstrap on the locked test.


In [ ]:
def compare_he_configurations(metrics_table, transformation_type="HEIR_REAL_HE_CONFIRMATORY"):
    real = metrics_table[metrics_table["transformation_type"] == transformation_type].copy()
    if real["configuration_id"].nunique() < 2:
        raise ValueError("H4 requires at least two distinct real HE configurations.")
    preferred = [
        "configuration_id", "plaintext_binary_accuracy", "transformed_binary_accuracy",
        "absolute_binary_accuracy_difference", "output_mae", "output_p95_abs_error",
        "float32_export_mae_vs_float64_plaintext", "he_mae_vs_float32_export",
        "he_p95_abs_error_vs_float32_export", "classification_disagreement_rate",
        "operational_action_disagreement_rate", "near_boundary_operational_disagreement_rate",
        "H2_status", "H3_status",
    ]
    cols = [c for c in preferred if c in real.columns]
    return real[cols].sort_values("configuration_id")

def paired_h4_configuration_evidence(detail_a, detail_b, config_a, config_b, reps, seed, confidence=0.95):
    a = detail_a.sort_values("sample_id").reset_index(drop=True)
    b = detail_b.sort_values("sample_id").reset_index(drop=True)
    if not np.array_equal(a["sample_id"].to_numpy(), b["sample_id"].to_numpy()):
        raise ValueError("H4 details are not aligned on identical sample IDs.")
    if "he_error_vs_float32_export" not in a or "he_error_vs_float32_export" not in b:
        raise ValueError("H4 requires decomposed incremental HE errors.")

    he_a = a["he_error_vs_float32_export"].to_numpy(float)
    he_b = b["he_error_vs_float32_export"].to_numpy(float)
    total_a = a["transformation_error"].to_numpy(float)
    total_b = b["transformation_error"].to_numpy(float)
    act_a = a["operational_action_disagreement"].astype(float).to_numpy()
    act_b = b["operational_action_disagreement"].astype(float).to_numpy()
    n = len(a)
    rng = np.random.default_rng(seed)
    d_he, d_total, d_action, d_he_p95 = [], [], [], []
    for _ in range(reps):
        idx = rng.integers(0, n, n)
        d_he.append(float(np.mean(he_b[idx] - he_a[idx])))
        d_total.append(float(np.mean(total_b[idx] - total_a[idx])))
        d_action.append(float(np.mean(act_b[idx] - act_a[idx])))
        d_he_p95.append(float(np.quantile(he_b[idx], 0.95) - np.quantile(he_a[idx], 0.95)))

    he_diff = he_b - he_a
    he_ci = percentile_ci(d_he, confidence)
    total_ci = percentile_ci(d_total, confidence)
    action_ci = percentile_ci(d_action, confidence)
    p95_ci = percentile_ci(d_he_p95, confidence)
    perm_p = paired_signflip_randomization_test(he_diff, H4_RANDOMIZATION_REPS, seed + 1)
    paired_sd = float(np.std(he_diff, ddof=1)) if n > 1 else 0.0
    effect = None if paired_sd <= np.finfo(float).eps else float(np.mean(he_diff) / paired_sd)
    return {
        "configuration_a": config_a,
        "configuration_b": config_b,
        "direction": "B minus A",
        "primary_endpoint": "delta_incremental_HE_MAE_vs_float32_export",
        "delta_incremental_he_mae": float(np.mean(he_diff)),
        "delta_incremental_he_mae_ci": he_ci,
        "paired_signflip_randomization_p_two_sided": perm_p,
        "standardized_paired_mean_difference": effect,
        "primary_H4_difference_detected_by_bootstrap_ci": bool(he_ci[0] > 0 or he_ci[1] < 0),
        "secondary_delta_incremental_he_p95_error": float(np.quantile(he_b, 0.95) - np.quantile(he_a, 0.95)),
        "secondary_delta_incremental_he_p95_error_ci": p95_ci,
        "secondary_delta_total_output_mae": float(np.mean(total_b - total_a)),
        "secondary_delta_total_output_mae_ci": total_ci,
        "secondary_delta_operational_disagreement_rate": float(np.mean(act_b - act_a)),
        "secondary_delta_operational_disagreement_rate_ci": action_ci,
        "bootstrap_reps": int(reps),
        "randomization_reps": int(H4_RANDOMIZATION_REPS),
        "confidence_level": float(confidence),
    }

print("H4 robustness helpers ready for the two pre-specified configurations.")


## 20. Confirmatory execution and analysis code (defined before freeze)

The functions that execute frozen binaries, validate execution bundles, analyze the locked test, and finalize H1–H4 are defined **before** the confirmatory freeze. Their bytecode fingerprints are stored in the freeze record, so confirmatory replay fails if this critical logic changes.


In [ ]:
CONFIRMATORY_RESULTS_DIR = RUN_DIR / "confirmatory"
CONFIRMATORY_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def _require_confirmatory_gate():
    if STUDY_MODE != "CONFIRMATORY" or not LOCKED_TEST_ACCESS_GRANTED or not LOCKED_TEST_MATERIALIZED:
        raise RuntimeError("Confirmed protocol and locked-test gate required.")
    validate_confirmatory_freeze(confirmatory_freeze)

def _frozen_configuration_ids():
    _require_confirmatory_gate()
    return list(confirmatory_freeze["frozen_real_he_configuration_ids"])

def validate_execution_bundle(configuration_id, execution_dir):
    _require_confirmatory_gate()
    configuration_id = checked_id(configuration_id)
    if configuration_id not in confirmatory_freeze["configurations"]:
        raise ValueError("Configuration is not frozen.")
    cfg = confirmatory_freeze["configurations"][configuration_id]
    directory = Path(execution_dir).resolve()
    record = json.loads((directory / "execution.json").read_text())
    if record["configuration_id"] != configuration_id or record["freeze_sha256"] != EXPECTED_CONFIRMATORY_FREEZE_SHA256:
        raise RuntimeError("Execution does not belong to this freeze/configuration.")
    expected_ids = [int(v) for v in test_idx]
    if record["sample_ids"] != expected_ids or record["repeats"] != cfg["repeats"] or record["return_code"] != 0:
        raise RuntimeError("Execution sample order, repetitions or return code differs.")
    if record["executable_sha256"] != cfg["artifacts"][cfg["executable"]]:
        raise RuntimeError("Execution binary is not the frozen binary.")
    for file, field in [("inputs.txt", "input_sha256"), ("stdout.log", "stdout_sha256"), ("stderr.log", "stderr_sha256"), ("scores.csv", "scores_sha256")]:
        if sha256_file(directory / file) != record[field]:
            raise RuntimeError(f"Execution artifact changed: {file}")
    rows = [line.split() for line in (directory / "inputs.txt").read_text().splitlines()]
    if [int(row[0]) for row in rows] != expected_ids:
        raise RuntimeError("Input sample order differs from the frozen test split.")
    values = np.asarray([[float(v) for v in row[1:]] for row in rows], dtype=np.float32)
    if not np.array_equal(values, X_test.to_numpy(dtype=np.float32)):
        raise RuntimeError("Execution input values differ from the locked test.")
    frame, _, runtime_parameters = parse_he_output((directory / "stdout.log").read_text(), expected_ids, cfg["repeats"])
    saved = pd.read_csv(directory / "scores.csv", float_precision="round_trip")
    if not frame.equals(saved):
        raise RuntimeError("Parsed results differ from recorded output.")
    frozen_ring = cfg["security_parameters"].get("ring_dimension")
    current_ring = runtime_parameters.get("PARAM_RING_DIMENSION")
    if frozen_ring is not None and current_ring is not None and int(frozen_ring) != int(current_ring):
        raise RuntimeError("Runtime ring dimension differs from the frozen calibration executable evidence.")
    return frame, record

def evaluate_confirmatory_configuration(configuration_id, execution_dir):
    frame, execution = validate_execution_bundle(configuration_id, execution_dir)
    z_arr = ordered_scores(frame, test_idx, repeat=0)
    metrics, detail = evaluate_decision_invariance(
        y_test, z_test, z_arr, NEAR_BOUNDARY_CUTOFF, "HEIR_REAL_HE_CONFIRMATORY", configuration_id
    )
    detail["sample_id"] = [int(v) for v in test_idx]
    z_export_f32 = float32_export_score(X_test)
    detail["z_plain_float32_export"] = z_export_f32
    detail["float32_export_error_vs_float64"] = np.abs(z_export_f32 - z_test)
    detail["he_error_vs_float32_export"] = np.abs(z_arr - z_export_f32)
    metrics.update({
        "float32_export_mae_vs_float64_plaintext": float(np.mean(detail["float32_export_error_vs_float64"])),
        "float32_export_p95_abs_error_vs_float64_plaintext": float(np.quantile(detail["float32_export_error_vs_float64"], 0.95)),
        "he_mae_vs_float32_export": float(np.mean(detail["he_error_vs_float32_export"])),
        "he_p95_abs_error_vs_float32_export": float(np.quantile(detail["he_error_vs_float32_export"], 0.95)),
        "he_p99_abs_error_vs_float32_export": float(np.quantile(detail["he_error_vs_float32_export"], 0.99)),
        "runtime_wall_seconds": float(execution["wall_seconds"]),
        "context_setup_ms": execution["timing"]["TIMING_CONTEXT_SETUP_MS"],
        "keygen_ms": execution["timing"]["TIMING_KEYGEN_MS"],
        "context_configure_ms": execution["timing"]["TIMING_CONTEXT_CONFIGURE_MS"],
    })
    boundary_sensitivity = boundary_cutoff_sensitivity(detail, NEAR_BOUNDARY_CUTOFFS)
    policy_sensitivity = policy_threshold_sensitivity(z_test, z_arr, POLICY_SENSITIVITY_PROB_PAIRS)
    bootstrap = paired_bootstrap_evidence(
        detail, np.asarray(y_test), reps=BOOTSTRAP_REPS_CONFIRMATORY,
        seed=SEED + 5000 + sum(ord(ch) for ch in configuration_id), confidence=CI_LEVEL
    )
    h1 = evaluate_h1_against_frozen_criteria(metrics, confirmatory_freeze)
    repetitions, all_detail, repeated_summary = repeated_decision_evidence(
        frame, test_idx, z_test, y_test, configuration_id, "HEIR_REAL_HE_CONFIRMATORY"
    )
    metrics.update({"primary_repeat": 0, "freeze_sha256": EXPECTED_CONFIRMATORY_FREEZE_SHA256})
    config_dir = CONFIRMATORY_RESULTS_DIR / configuration_id
    config_dir.mkdir(exist_ok=True)
    if (config_dir / "metrics.json").exists():
        raise FileExistsError("This run already has evaluated results for this configuration.")
    np.save(config_dir / "locked_test_he_scores.npy", z_arr)
    detail.to_csv(config_dir / "locked_test_detail.csv", index=False)
    repetitions.to_csv(config_dir / "per_repeat_metrics.csv", index=False)
    all_detail.to_csv(config_dir / "all_repetitions_detail.csv", index=False)
    boundary_sensitivity.to_csv(config_dir / "boundary_cutoff_sensitivity.csv", index=False)
    policy_sensitivity.to_csv(config_dir / "policy_threshold_sensitivity.csv", index=False)
    for name, value in [("metrics", metrics), ("bootstrap", bootstrap), ("H1_evaluation", h1), ("repeated_decisions", repeated_summary), ("runtime_record", execution)]:
        write_json_exclusive(config_dir / f"{name}.json", value)
    analysis_files = {path.name: sha256_file(path) for path in config_dir.iterdir() if path.is_file()}
    write_json_exclusive(config_dir / "analysis_integrity.json", {
        "freeze_sha256": EXPECTED_CONFIRMATORY_FREEZE_SHA256,
        "execution_dir": str(Path(execution_dir).resolve().relative_to(ROOT.resolve())),
        "files": analysis_files,
    })
    return metrics, detail, bootstrap, h1

def verify_analysis_results(configuration_id):
    directory = CONFIRMATORY_RESULTS_DIR / configuration_id
    integrity = json.loads((directory / "analysis_integrity.json").read_text())
    if integrity["freeze_sha256"] != EXPECTED_CONFIRMATORY_FREEZE_SHA256:
        raise RuntimeError("Analysis belongs to another freeze.")
    for name, expected in integrity["files"].items():
        if Path(name).name != name or sha256_file(directory / name) != expected:
            raise RuntimeError("Analysis file is modified or invalid.")
    validate_execution_bundle(configuration_id, artifact_path(integrity["execution_dir"]))

def run_all_confirmatory_configurations():
    _require_confirmatory_gate()
    for cid in _frozen_configuration_ids():
        cfg = confirmatory_freeze["configurations"][cid]
        destination = CONFIRMATORY_RESULTS_DIR / cid / "execution"
        if destination.exists():
            raise FileExistsError("Execution directory already exists. Use a new run; preserve prior results.")
        verify_artifacts(cfg["artifacts"])
        frame, execution = execute_he_binary(
            artifact_path(cfg["executable"]), X_test.to_numpy(dtype=np.float32), test_idx,
            cfg["repeats"], destination, artifact_path(cfg["runtime_lib_dir"])
        )
        execution.update({"configuration_id": cid, "freeze_sha256": EXPECTED_CONFIRMATORY_FREEZE_SHA256})
        verify_artifacts(cfg["artifacts"])
        write_json_exclusive(destination / "execution.json", execution)
        evaluate_confirmatory_configuration(cid, destination)
    return finalize_confirmatory_study()

def finalize_confirmatory_study():
    _require_confirmatory_gate()
    frozen_ids = _frozen_configuration_ids()
    rows, missing = [], []
    details = {}
    for config_id in frozen_ids:
        config_dir = CONFIRMATORY_RESULTS_DIR / config_id
        metrics_path = config_dir / "metrics.json"
        h1_path = config_dir / "H1_evaluation.json"
        bootstrap_path = config_dir / "bootstrap.json"
        detail_path = config_dir / "locked_test_detail.csv"
        if not all(p.exists() for p in [metrics_path, h1_path, bootstrap_path, detail_path]):
            missing.append(config_id)
            continue
        verify_analysis_results(config_id)
        metrics = json.loads(metrics_path.read_text())
        h1 = json.loads(h1_path.read_text())
        bootstrap = json.loads(bootstrap_path.read_text())
        details[config_id] = pd.read_csv(detail_path)
        rows.append({
            **metrics,
            "H1_binary_predictions_preserved": h1["binary_predictions_preserved"],
            "H1_operational_gap_present": h1["operational_gap_present"],
            "H1_strict_coexistence_flag": h1["H1_strict_coexistence_flag"],
            "H1_condition_met": h1["H1_condition_met"],
            "H3_delta_auc_bootstrap_mean": bootstrap["H3_delta_auc_mean"],
            "H3_delta_auc_ci_low": bootstrap["H3_delta_auc_ci"][0],
            "H3_delta_auc_ci_high": bootstrap["H3_delta_auc_ci"][1],
        })
    if missing:
        raise RuntimeError("Missing confirmatory results for: " + ", ".join(missing))

    summary = pd.DataFrame(rows).sort_values("configuration_id").reset_index(drop=True)
    summary["H2_fisher_p_holm"] = holm_adjust(summary["H2_fisher_p_one_sided"].to_numpy(float))
    summary["H2_reject_familywise_0_05"] = (summary["H2_fisher_p_holm"] <= FAMILYWISE_ALPHA).fillna(False)
    summary.to_csv(CONFIRMATORY_RESULTS_DIR / "confirmatory_summary.csv", index=False)
    paper_cols = [
        "configuration_id", "transformed_binary_accuracy", "absolute_binary_accuracy_difference",
        "output_mae", "output_p95_abs_error", "he_mae_vs_float32_export", "he_p95_abs_error_vs_float32_export",
        "classification_disagreement_rate", "operational_action_disagreement_rate",
        "operational_action_disagreement_ci_low", "operational_action_disagreement_ci_high",
        "operational_action_disagreement_upper_bound_one_sided",
        "near_boundary_operational_disagreement_rate", "H2_status", "H2_fisher_p_one_sided", "H2_fisher_p_holm",
        "H3_status", "H3_auc_raw_error", "H3_auc_error_to_margin_ratio",
        "H3_delta_auc_bootstrap_mean", "H3_delta_auc_ci_low", "H3_delta_auc_ci_high",
        "H1_strict_coexistence_flag",
    ]
    paper_table = summary[paper_cols].copy()
    paper_table.to_csv(CONFIRMATORY_RESULTS_DIR / "paper_results_table.csv", index=False)

    h4_table = compare_he_configurations(summary, transformation_type="HEIR_REAL_HE_CONFIRMATORY")
    h4_table.to_csv(CONFIRMATORY_RESULTS_DIR / "H4_configuration_comparison.csv", index=False)
    if len(frozen_ids) != 2:
        raise RuntimeError("v1.3 H4 confirmatory analysis is pre-specified for exactly two configurations.")
    h4 = paired_h4_configuration_evidence(
        details[frozen_ids[0]], details[frozen_ids[1]], frozen_ids[0], frozen_ids[1],
        reps=BOOTSTRAP_REPS_CONFIRMATORY, seed=SEED + 9000, confidence=CI_LEVEL
    )
    (CONFIRMATORY_RESULTS_DIR / "H4_paired_bootstrap.json").write_text(json.dumps(h4, indent=2))

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(summary["he_mae_vs_float32_export"], summary["operational_action_disagreement_rate"], s=70)
    for _, row in summary.iterrows():
        ax.annotate(row["configuration_id"], (row["he_mae_vs_float32_export"], row["operational_action_disagreement_rate"]), xytext=(5,5), textcoords="offset points", fontsize=8)
    ax.set_xlabel("Incremental HE MAE vs float32-export plaintext")
    ax.set_ylabel("Operational-action disagreement rate")
    ax.set_title("Confirmatory CKKS configurations: numeric fidelity vs operational gap")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(CONFIRMATORY_RESULTS_DIR / "confirmatory_fidelity_vs_operational_gap.png", dpi=300, bbox_inches="tight")
    fig.savefig(CONFIRMATORY_RESULTS_DIR / "confirmatory_fidelity_vs_operational_gap.svg", bbox_inches="tight")
    plt.show()

    final_record = {
        "n_frozen_configurations": int(len(summary)),
        "all_frozen_configurations_evaluated": True,
        "H1_any_configuration_strict_coexistence_observed": bool(summary["H1_strict_coexistence_flag"].any()),
        "H2_status": {row["configuration_id"]: row["H2_status"] for _, row in summary.iterrows()},
        "H2_holm_adjusted_p": {row["configuration_id"]: (None if not np.isfinite(row["H2_fisher_p_holm"]) else float(row["H2_fisher_p_holm"])) for _, row in summary.iterrows()},
        "H3_status": {row["configuration_id"]: row["H3_status"] for _, row in summary.iterrows()},
        "locked_test_rare_event_resolution": rare_event_planning,
        "H4_paired_evidence": h4,
        "freeze_sha256": EXPECTED_CONFIRMATORY_FREEZE_SHA256,
        "transformation_type": "HEIR_REAL_HE_CONFIRMATORY",
    }
    (CONFIRMATORY_RESULTS_DIR / "final_confirmatory_record.json").write_text(json.dumps(final_record, indent=2))
    display(paper_table)
    return summary, paper_table, h4_table, final_record


## 21. Automatic configuration registration and confirmatory freeze

All pre-specified calibration configurations must execute successfully. The notebook then registers **all of them** and freezes the entire set, preventing result-based configuration selection. H1 uses a direct empirical rule rather than post-hoc numerical tolerances.


In [ ]:
def register_he_configuration(configuration_id):
    if STUDY_MODE != "PILOT" or CONFIRMATORY_FREEZE_PATH.exists():
        raise RuntimeError("Registration is allowed only in unfrozen PILOT mode.")
    configuration_id = checked_id(configuration_id)
    source = ROOT / "artifacts" / configuration_id
    completed = json.loads((source / "configuration_executed.json").read_text())
    if completed["status"] != "EXECUTED" or completed["stage"] != "CALIBRATION_PILOT":
        raise ValueError("A successfully executed calibration pilot is required.")

    expected_spec = next((spec for spec in HE_CONFIGURATION_SPECS if spec["configuration_id"] == configuration_id), None)
    if expected_spec is None:
        raise ValueError("Configuration was not pre-specified.")
    if completed.get("requested_configuration") != expected_spec:
        raise RuntimeError("Executed configuration differs from the pre-specified configuration.")
    if int(completed.get("pilot_samples", -1)) != len(cal_idx) or CKKS_PILOT_SAMPLE_LIMIT is not None:
        raise RuntimeError("A complete-study freeze requires the full calibration split, not smoke-test sampling.")

    params = completed["normalized_parameters"]
    for key in ["ring_dimension", "multiplicative_depth", "scaling_mod_size", "first_mod_size"]:
        value = params.get(key)
        minimum = 0 if key == "multiplicative_depth" else 1
        if isinstance(value, bool) or not isinstance(value, int) or value < minimum:
            raise ValueError(f"Invalid normalized CKKS parameter {key}: {value}")
    ring = params["ring_dimension"]
    if ring < 2 or ring & (ring - 1):
        raise ValueError("Ring dimension must be a power of two.")

    # Security level is not fabricated. An explicit generated SetSecurityLevel is
    # recorded when present; otherwise the study records the absence and makes no
    # independent security-strength claim.
    security_claim_status = "EXPLICIT_GENERATED_LEVEL" if params.get("security_level") else "NO_INDEPENDENT_SECURITY_LEVEL_CLAIM"

    runtime_dir = source / "runtime_libs"
    runtime_dir.mkdir(exist_ok=True)
    candidates = {}
    for directory in completed["openfhe_library_dirs"]:
        for path in Path(directory).glob("libOPENFHE*.so*"):
            if path.name in candidates and sha256_file(path) != sha256_file(candidates[path.name]):
                raise RuntimeError(f"Ambiguous OpenFHE library: {path.name}")
            candidates[path.name] = path
    for family in ["core", "pke", "binfhe"]:
        if not any(name.startswith(f"libOPENFHE{family}.so") for name in candidates):
            raise RuntimeError(f"Missing OpenFHE {family} shared library.")
    for name, path in candidates.items():
        shutil.copy2(path, runtime_dir / name)

    ldd_proc = subprocess.run(["ldd", str(source / "affine_score_pilot")], capture_output=True, text=True, check=False)
    runtime_dependency_path = source / "runtime_dependencies.txt"
    runtime_dependency_path.write_text((ldd_proc.stdout or "") + ("\nSTDERR:\n" + ldd_proc.stderr if ldd_proc.stderr else ""), encoding="utf-8")
    if ldd_proc.returncode != 0 or "not found" in (ldd_proc.stdout or ""):
        raise RuntimeError("Frozen executable has unresolved dynamic dependencies.")

    required_names = [
        "affine_score_input.mlir", "affine_score_openfhe.mlir", "affine_score.cpp", "affine_score.h",
        "affine_score_pilot_main.cpp", "affine_score_pilot", "CMakeLists.codegen_probe.txt",
        "cmake_codegen_configure.log", "cmake_codegen_build.log", "CMakeLists.runtime.txt",
        "cmake_runtime_configure.log", "cmake_runtime_build.log", "pilot_ldd_before_execution.txt",
        "compiler_parameter_evidence.json", "optional_heir_emitters.json", "configuration_executed.json",
        "runtime_dependencies.txt", "calibration_boundary_cutoff_sensitivity.csv",
        "calibration_policy_threshold_sensitivity.csv", "boundary_stress_detail.csv",
        "boundary_stress_summary.json", "boundary_stress_by_delta.csv",
    ]
    paths = [source / name for name in required_names]
    missing = [p.name for p in paths if not p.is_file()]
    if missing:
        raise RuntimeError(f"Required freeze artifacts are missing for {configuration_id}: {missing}")
    paths += [p for p in runtime_dir.iterdir() if p.is_file()]
    hashes = {str(path.relative_to(ROOT)): sha256_file(path) for path in paths}

    record = {
        "configuration_id": configuration_id,
        "status": "REGISTERED_AFTER_CALIBRATION",
        "scheme": "CKKS", "backend": "OpenFHE",
        "requested_configuration": completed["requested_configuration"],
        "security_parameters": params,
        "security_claim_status": security_claim_status,
        "parameter_validation": "automatically normalized from generated C++/lowered MLIR/runtime CryptoContext evidence",
        "executable": str((source / "affine_score_pilot").relative_to(ROOT)),
        "runtime_lib_dir": str(runtime_dir.relative_to(ROOT)),
        "artifacts": hashes,
        "science_sha256": completed["science_sha256"],
        "repeats": completed["runtime_repeats"],
        "pilot_samples": int(completed["pilot_samples"]),
        "heir_pipeline": completed["heir_pipeline"],
    }
    if record["science_sha256"] != digest_json(scientific_snapshot()):
        raise RuntimeError("Calibration used different scientific settings.")
    write_json_exclusive(REGISTRY_DIR / f"{configuration_id}.json", record)
    return record

def confirmatory_code_fingerprints():
    critical = [
        sha256_file, verify_artifacts, parse_he_output, ordered_scores, repeated_decision_evidence,
        execute_he_binary, compare_he_configurations, paired_h4_configuration_evidence,
        boundary_cutoff_sensitivity, policy_threshold_sensitivity, holm_adjust,
        paired_signflip_randomization_test, float32_export_score,
        _require_confirmatory_gate, _frozen_configuration_ids, validate_execution_bundle,
        evaluate_confirmatory_configuration, verify_analysis_results,
        run_all_confirmatory_configurations, finalize_confirmatory_study,
        evaluate_h1_against_frozen_criteria,
    ]
    return transitive_code_manifest(critical)

def freeze_confirmatory_protocol(frozen_configuration_ids=None):
    if STUDY_MODE != "PILOT" or CONFIRMATORY_FREEZE_PATH.exists():
        raise RuntimeError("Freeze must be created once, in PILOT mode, in an unfrozen project.")
    if CKKS_PILOT_SAMPLE_LIMIT is not None:
        raise RuntimeError("Confirmatory freeze is forbidden after a smoke-test sample limit; rerun a full PILOT.")
    expected_ids = [spec["configuration_id"] for spec in HE_CONFIGURATION_SPECS]
    ids = expected_ids if frozen_configuration_ids is None else [checked_id(v) for v in frozen_configuration_ids]
    if ids != expected_ids:
        raise ValueError("All and only the pre-specified HE configurations must be frozen in their pre-specified order.")

    science = scientific_snapshot()
    configs = {}
    for cid in ids:
        path = REGISTRY_DIR / f"{cid}.json"
        if not path.is_file():
            raise RuntimeError(f"Pre-specified configuration is not registered: {cid}")
        record = json.loads(path.read_text())
        verify_artifacts(record["artifacts"])
        if record["configuration_id"] != cid or record["science_sha256"] != digest_json(science):
            raise RuntimeError("Registered configuration does not match current scientific state.")
        if record["repeats"] != CKKS_RUNTIME_REPEATS:
            raise RuntimeError("Repetition count differs from registered executable.")
        if int(record.get("pilot_samples", -1)) != len(cal_idx):
            raise RuntimeError("Frozen configuration was not evaluated on the full calibration split.")
        configs[cid] = record

    dependencies = dependency_versions()
    req_path = ROOT / "environment" / "confirmatory_requirements.txt"
    req_path.write_text("\n".join(f"{name}=={version}" for name, version in dependencies.items()) + "\n")
    environment_provenance = runtime_environment_provenance()
    environment_contract = environment_compatibility_contract(environment_provenance)
    environment_provenance_path = ROOT / "environment" / "confirmatory_environment_provenance.json"
    environment_provenance_path.write_text(json.dumps(environment_provenance, indent=2), encoding="utf-8")
    tracked = [
        ROOT / "models" / "plaintext_logistic_regression.joblib",
        ROOT / "models" / "plaintext_affine_model.npz",
        ROOT / "data" / "split_indices.csv",
        ROOT / "protocol" / "split_provenance.json",
        ROOT / "data" / "boundary_stress_features.csv",
        ROOT / "data" / "boundary_stress_metadata.csv",
        ROOT / "protocol" / "boundary_stress_plausibility.json",
        ROOT / "protocol" / "rare_event_planning.json",
        req_path,
        environment_provenance_path,
        ROOT / "protocol" / "protocol.json",
        ROOT / "protocol" / "frozen_operational_policy.json",
    ]
    freeze = {
        "schema_version": 6,
        "study_version": STUDY_VERSION,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "frozen_real_he_configuration_ids": ids,
        "configurations": configs,
        "science": science,
        "science_sha256": digest_json(science),
        "artifacts": {str(p.relative_to(ROOT)): sha256_file(p) for p in tracked},
        "dependencies": dependencies,
        "environment_provenance": environment_provenance,
        "environment_contract": environment_contract,
        "python_minor": list(sys.version_info[:2]),
        "machine": platform.machine(),
        "confirmatory_code": confirmatory_code_fingerprints(),
        "H1_criteria": {
            "binary_prediction_disagreement_count_must_equal": 0,
            "operational_action_disagreement_count_must_be_at_least": 1,
            "interpretation": (
                "strict coexistence flag only; aggregate binary accuracy difference and classification disagreement "
                "are reported separately without an application-independent non-inferiority margin"
            ),
        },
        "H2_rule": "not estimable if no operational disagreements or an empty near/far stratum",
        "H3_rule": (
            "diagnostic only; not estimable unless operational disagreement labels contain both classes; "
            "no reject/accept multiplicity claim"
        ),
        "H4_rule": "paired bootstrap and sign-flip randomization B-minus-A; primary endpoint delta incremental HE MAE vs float32 export",
        "robustness_plan": {
            "near_boundary_sensitivity_cutoffs": NEAR_BOUNDARY_CUTOFFS,
            "policy_sensitivity_probability_pairs": POLICY_SENSITIVITY_PROB_PAIRS,
            "boundary_stress_deltas": BOUNDARY_STRESS_DELTAS,
            "familywise_alpha": FAMILYWISE_ALPHA,
            "h4_randomization_reps": H4_RANDOMIZATION_REPS,
            "rare_event_planning": rare_event_planning,
            "stress_plausibility": BOUNDARY_STRESS_PLAUSIBILITY,
            "split_provenance_sha256": sha256_file(ROOT / "protocol" / "split_provenance.json"),
            "multiplicity_scope": {
                "H2_family": "Holm-adjusted across two primary configuration-specific tests",
                "H3": "diagnostic only",
                "H4": "separate pre-specified paired numerical endpoint",
                "omnibus_H1_to_H4_FWER_claim": False,
            },
        },
        "operational_policy": {
            "low_logit_threshold": OPERATIONAL_LOW_THRESHOLD,
            "high_logit_threshold": OPERATIONAL_HIGH_THRESHOLD,
            "near_boundary_cutoff": NEAR_BOUNDARY_CUTOFF,
        },
        "infrastructure": {
            "heir_py_version": HEIR_PY_VERSION,
            "heir_release_commit": HEIR_RELEASE_COMMIT,
            "heir_expected_openfhe_bcr_version": HEIR_OPENFHE_BCR_VERSION,
            "openfhe_git_ref": OPENFHE_GIT_REF,
            "openfhe_commit": resolved_openfhe_sha,
        },
    }
    write_json_exclusive(CONFIRMATORY_FREEZE_PATH, freeze)
    digest = sha256_file(CONFIRMATORY_FREEZE_PATH)
    (ROOT / "protocol" / "confirmatory_freeze.sha256").write_text(digest + "\n")
    print("CONFIRMATORY FREEZE CREATED")
    print("Preserve this digest independently:", digest)
    return freeze

def validate_confirmatory_freeze(record):
    if record.get("schema_version") != 6 or record.get("study_version") != STUDY_VERSION:
        raise RuntimeError("Freeze schema/study version differs; a new pilot freeze is required.")
    if not EXPECTED_CONFIRMATORY_FREEZE_SHA256:
        raise RuntimeError("Set EXPECTED_CONFIRMATORY_FREEZE_SHA256.")
    if sha256_file(CONFIRMATORY_FREEZE_PATH) != EXPECTED_CONFIRMATORY_FREEZE_SHA256:
        raise RuntimeError("Freeze file differs from the independently preserved digest.")
    verify_artifacts(record["artifacts"])
    if record.get("confirmatory_code") != confirmatory_code_fingerprints():
        raise RuntimeError("Critical confirmatory code differs from the freeze.")
    if record["science_sha256"] != digest_json(scientific_snapshot()):
        raise RuntimeError("Data, split, model, policy, HE specs, seed, or analysis plan differs from the freeze.")
    if dependency_versions() != record["dependencies"]:
        raise RuntimeError("Scientific/HEIR dependency versions differ from the freeze.")
    current_environment = runtime_environment_provenance()
    current_contract = environment_compatibility_contract(current_environment)
    if current_contract != record.get("environment_contract"):
        raise RuntimeError(
            "Strict runtime compatibility contract differs from the pilot. "
            f"Frozen={record.get('environment_contract')} current={current_contract}"
        )
    descriptive_environment_differences = {
        key: {"pilot": record["environment_provenance"].get(key), "current": current_environment.get(key)}
        for key in sorted(set(record["environment_provenance"]) | set(current_environment))
        if record["environment_provenance"].get(key) != current_environment.get(key)
    }
    if descriptive_environment_differences:
        report_path = RUN_DIR / "confirmatory_environment_differences.json"
        report_path.write_text(json.dumps(descriptive_environment_differences, indent=2), encoding="utf-8")
        print(
            "WARNING: non-contract runtime provenance differs from the pilot; "
            f"details recorded at {report_path}. Dependency and strict compatibility checks still apply."
        )
    if importlib_metadata.version("heir_py") != record["infrastructure"]["heir_py_version"]:
        raise RuntimeError("Executing HEIR version differs from the freeze.")
    for cid in record["frozen_real_he_configuration_ids"]:
        cfg = record["configurations"][cid]
        verify_artifacts(cfg["artifacts"])
        if cfg["science_sha256"] != record["science_sha256"] or cfg["repeats"] != CKKS_RUNTIME_REPEATS:
            raise RuntimeError("Frozen configuration differs from the scientific state.")
    return True

def evaluate_h1_against_frozen_criteria(metrics, freeze_record):
    criteria = freeze_record["H1_criteria"]
    binary_preserved = metrics["classification_disagreement_count"] == criteria["binary_prediction_disagreement_count_must_equal"]
    operational_gap = metrics["operational_action_disagreement_count"] >= criteria["operational_action_disagreement_count_must_be_at_least"]
    strict_flag = bool(binary_preserved and operational_gap)
    return {
        "binary_predictions_preserved": bool(binary_preserved),
        "operational_gap_present": bool(operational_gap),
        "H1_strict_coexistence_flag": strict_flag,
        "H1_condition_met": strict_flag,  # backward-compatible alias
        "aggregate_binary_accuracy_difference": float(metrics["absolute_binary_accuracy_difference"]),
        "classification_disagreement_rate": float(metrics["classification_disagreement_rate"]),
        "note": (
            "Strict finite-sample coexistence flag only. Aggregate predictive fidelity is reported descriptively; "
            "absence of observed disagreement is not universal invariance and no arbitrary non-inferiority margin is imposed."
        ),
    }

if STUDY_MODE == "PILOT":
    registered = []
    for spec in HE_CONFIGURATION_SPECS:
        registered.append(register_he_configuration(spec["configuration_id"]))
    print("Automatically registered configurations:", [r["configuration_id"] for r in registered])
    if AUTO_FREEZE_AFTER_PILOT:
        confirmatory_freeze = freeze_confirmatory_protocol()
else:
    print("CONFIRMATORY: registration/freeze creation skipped; existing freeze will be validated.")


### Automatic transition after calibration
No cryptographic parameters or configuration IDs need to be typed manually. The pilot registers both pre-specified successful configurations, freezes all of them, and prints the digest that must be preserved independently for the confirmatory run.


## 22. Locked-test gate

No locked-test summary is produced during PILOT. A clean confirmatory run may access the locked test only after the frozen bundle is restored and the independently preserved freeze digest is verified. The gate is evaluated before any locked-test plaintext summary or transformed score is produced.


In [ ]:

LOCKED_TEST_ACCESS_GRANTED = False
confirmatory_freeze = None

if STUDY_MODE != "CONFIRMATORY":
    print("LOCKED TEST SEALED: current run is PILOT.")
elif not CONFIRMATORY_FREEZE_PATH.exists():
    print(
        "LOCKED TEST SEALED: confirmatory_freeze.json is absent. "
        "Preserve/import the pilot freeze before confirmatory execution."
    )
else:
    confirmatory_freeze = json.loads(
        CONFIRMATORY_FREEZE_PATH.read_text(encoding="utf-8")
    )

    validate_confirmatory_freeze(confirmatory_freeze)

    LOCKED_TEST_ACCESS_GRANTED = True
    print("LOCKED TEST ACCESS GRANTED under the frozen confirmatory protocol.")



if LOCKED_TEST_ACCESS_GRANTED:
    # Materialize locked-test features/labels for the first time only now.
    X_test = X.iloc[test_idx].copy()
    y_test = y.iloc[test_idx].copy()
    z_test = affine_score(X_test)
    LOCKED_TEST_MATERIALIZED = True

    # Verify exported affine equivalence on the locked test only after the gate.
    locked_test_equivalence_error = float(
        np.max(
            np.abs(
                model.decision_function(X_test)
                - z_test
            )
        )
    )
    assert locked_test_equivalence_error <= 1e-10

    locked_plain_class = (
        z_test >= CLASSIFICATION_THRESHOLD
    ).astype(int)

    locked_plain_actions = apply_operational_policy(z_test)
    locked_plain_margin = operational_margin(z_test)

    locked_plain_baseline = {
        "n_locked_test": int(len(z_test)),
        "plaintext_binary_accuracy": float(
            accuracy_score(y_test, locked_plain_class)
        ),
        "locked_test_pipeline_vs_affine_max_error": (
            locked_test_equivalence_error
        ),
        "action_0_count": int(
            np.sum(locked_plain_actions == ACTION_0)
        ),
        "review_count": int(
            np.sum(locked_plain_actions == ACTION_REVIEW)
        ),
        "action_1_count": int(
            np.sum(locked_plain_actions == ACTION_1)
        ),
        "event_rate_giving_95pct_probability_of_at_least_one_event": event_rate_for_at_least_one(len(z_test), 0.95),
        "near_boundary_count_using_frozen_calibration_cutoff": int(
            np.sum(
                locked_plain_margin
                <= confirmatory_freeze["operational_policy"][
                    "near_boundary_cutoff"
                ]
            )
        ),
    }

    (
        ROOT
        / "results"
        / f"locked_plaintext_baseline_{RUN_ID}.json"
    ).write_text(
        json.dumps(locked_plain_baseline, indent=2),
        encoding="utf-8",
    )

    print(json.dumps(locked_plain_baseline, indent=2))


### Confirmatory execution

In `CONFIRMATORY` mode, the frozen execution/analysis functions from section 20 are invoked only after the locked-test gate is validated.


## 23. Traceable execution and final analysis

All frozen configurations execute automatically in CONFIRMATORY mode after validation.


The implementation itself is defined before the freeze (section 20) so its bytecode can be included in the confirmatory integrity record. This section contains the execution workflow and outputs.


In [ ]:
# Definitions are frozen earlier in section 20; intentionally not redefined here.


### Result registration

`evaluate_confirmatory_configuration` validates an execution directory against the freeze and sample IDs before producing metrics.


In [ ]:
# The evaluation implementation is defined with the execution-bundle validator above.


### Exploratory score-file utility

The generic reader is retained for development only. Its output is not accepted as confirmatory evidence.


In [ ]:
def load_confirmatory_scores(path):
    """Load a 1-D .npy, headerless text/CSV, or single-column CSV with a header."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".npy":
        scores = np.load(path, allow_pickle=False)
    else:
        # header=None preserves observation zero in files without column names.
        frame = pd.read_csv(path, header=None, dtype=str, skip_blank_lines=True)
        if frame.shape[1] != 1 or frame.empty:
            raise ValueError("Score file must contain exactly one non-empty column.")
        column = frame.iloc[:, 0]
        try:
            float(column.iloc[0])
        except (TypeError, ValueError):
            column = column.iloc[1:]  # optional single header row
        scores = pd.to_numeric(column, errors="raise").to_numpy(dtype=float)
    scores = np.asarray(scores, dtype=float)
    if scores.ndim == 2 and scores.shape[1] == 1:
        scores = scores[:, 0]
    if scores.ndim != 1 or scores.size == 0 or not np.isfinite(scores).all():
        raise ValueError("Scores must be a non-empty, finite, one-dimensional vector.")
    return scores



### 23.3 Final H1–H4 summary and paper-ready tables

Run this only after every frozen configuration has been evaluated.


In [ ]:
# finalize_confirmatory_study() is defined and fingerprinted in section 20.


### Run the frozen study

In CONFIRMATORY mode the following cell executes all frozen configurations and finalizes
the tables. It stops on any integrity or execution failure. In PILOT mode no locked-test
execution occurs. Inspect all results before interpreting H1–H4.


In [ ]:
if STUDY_MODE == "CONFIRMATORY":
    confirmatory_outputs = run_all_confirmatory_configurations()
else:
    if not CONFIRMATORY_FREEZE_PATH.is_file():
        raise RuntimeError("PILOT reached the end without a confirmatory freeze.")
    print("PILOT COMPLETE AND FROZEN.")
    print("Freeze digest:", sha256_file(CONFIRMATORY_FREEZE_PATH))
    print("Next: export/download the bundle below, preserve the digest independently, then run CONFIRMATORY in a fresh session.")


## 24. Open-science metadata


In [ ]:
readme = f'''# {PROJECT_NAME}

Open-science artifact for studying preservation of downstream operational decisions under real HEIR/OpenFHE CKKS transformation.

## Study version
{STUDY_VERSION}

## Study mode
{STUDY_MODE}

## Author
{AUTHOR} — {AFFILIATION}

## Scientific design
- provenance-preserving v1.0 train / calibration / locked-test separation
- three-action operational policy
- calibration-frozen near-boundary cutoff
- two pre-specified CKKS precision configurations with no post-hoc selection
- full calibration execution required before confirmatory freeze
- exact binomial and paired bootstrap uncertainty
- split-provenance hash and calibration-only stress plausibility diagnostics
- zero-event H2/H3 estimability guards
- automatic compiler/runtime parameter capture
- frozen data/model/split/configuration hashes
- transitive notebook-code manifest for scientific and confirmatory logic
- rich runtime provenance snapshot plus strict compatibility contract
- identified samples and per-repetition decisions
- frozen binaries with bundled OpenFHE runtime libraries
- dependency closure pinned for confirmatory replay

## Evidence labels
- SIMULATED_METHOD_VALIDATION: methodology development only
- HEIR_REAL_HE_PILOT_CALIBRATION: real-HE calibration/feasibility only
- HEIR_REAL_HE_CONFIRMATORY: frozen locked-test confirmatory evidence

The locked test is the original v1.0 held-out subset, not an external validation cohort. External replication remains a separate validity objective.

## Confirmatory replay
Restore the complete pilot bundle, set STUDY_MODE=CONFIRMATORY, provide RESTORE_BUNDLE_ZIP and the independently preserved EXPECTED_CONFIRMATORY_FREEZE_SHA256, then run from the first cell.

## Security-metadata note
The artifact records explicit generated security settings when present. If HEIR-generated C++ does not explicitly set a security level, the artifact records that absence and does not make an independent cryptographic-security-strength claim.
'''
if LICENSE_CODE:
    readme += f"\n## License\n{LICENSE_CODE}\n"
if REPOSITORY_URL:
    readme += f"\n## Repository\n{REPOSITORY_URL}\n"
missing_open_science_metadata = [
    name for name, value in [("REPOSITORY_URL", REPOSITORY_URL), ("LICENSE_CODE", LICENSE_CODE)] if not value
]
if missing_open_science_metadata:
    readme += (
        "\n## Publication metadata status\n"
        "The experimental artifact is executable, but the following publication metadata must be supplied "
        "before public release: " + ", ".join(missing_open_science_metadata) + ".\n"
    )
    print("OPEN-SCIENCE METADATA WARNING:", ", ".join(missing_open_science_metadata), "not yet supplied.")
(ROOT / "README.md").write_text(readme, encoding="utf-8")

citation_lines = [
    "cff-version: 1.2.0",
    'message: "If you use this research artifact, please cite the associated paper and repository release."',
    'title: "Robust Decision Invariance under Homomorphic Transformation"',
    f'version: "{STUDY_VERSION}"',
    "authors:",
    '  - family-names: "Figueira"',
    '    given-names: "Larissa de Oliveira"',
]
if REPOSITORY_URL:
    citation_lines.append(f'repository-code: "{REPOSITORY_URL}"')
if LICENSE_CODE:
    citation_lines.append(f'license: "{LICENSE_CODE}"')
(ROOT / "CITATION.cff").write_text("\n".join(citation_lines) + "\n", encoding="utf-8")

run_metadata = {
    "run_id": RUN_ID,
    "study_version": STUDY_VERSION,
    "study_mode": STUDY_MODE,
    "author": AUTHOR,
    "affiliation": AFFILIATION,
    "seed": SEED,
    "he_configuration_specs": HE_CONFIGURATION_SPECS,
    "heir_py_spec_requested": HEIR_PY_SPEC,
    "heir_release_commit": HEIR_RELEASE_COMMIT,
    "heir_expected_openfhe_bcr_version": HEIR_OPENFHE_BCR_VERSION,
    "heir_py_version_resolved": environment.get("heir_py_version"),
    "openfhe_requested_ref": OPENFHE_GIT_REF,
    "confirmatory_freeze_exists": CONFIRMATORY_FREEZE_PATH.exists(),
    "locked_test_access_granted": LOCKED_TEST_ACCESS_GRANTED,
    "locked_test_materialized": LOCKED_TEST_MATERIALIZED,
    "openfhe_resolved_commit": resolved_openfhe_sha,
    "git_state": git_state,
    "near_boundary_cutoff": NEAR_BOUNDARY_CUTOFF,
    "operational_low_threshold": OPERATIONAL_LOW_THRESHOLD,
    "operational_high_threshold": OPERATIONAL_HIGH_THRESHOLD,
    "bootstrap_reps": BOOTSTRAP_REPS,
}
(RUN_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
print("README.md, CITATION.cff, and run metadata created.")


## 25. Whole-project SHA-256 manifest


In [ ]:
manifest_path = ROOT / "MANIFEST.sha256.json"
entries = []
for path in sorted(ROOT.rglob("*")):
    if not path.is_file() or path == manifest_path:
        continue
    entries.append({
        "path": str(path.relative_to(ROOT)),
        "bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    })
manifest = {
    "project_name": PROJECT_NAME,
    "study_version": STUDY_VERSION,
    "study_mode": STUDY_MODE,
    "run_id": RUN_ID,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "manifest_excludes_itself": True,
    "git_state": git_state,
    "entries": entries,
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Manifest entries:", len(entries))
print("Manifest:", manifest_path)


## 26. Export complete open-science bundle


In [ ]:

bundle_base = Path("/content") / (
    f"{PROJECT_NAME}_{STUDY_VERSION}_{RUN_ID}"
)

archive_path = shutil.make_archive(
    str(bundle_base),
    "zip",
    root_dir=ROOT,
)

print("Open-science bundle:")
print(archive_path)


# Before interpreting the study

Use the evidence labels strictly:

- **SIMULATED_METHOD_VALIDATION**: calibration-only method-development evidence.
- **HEIR_REAL_HE_PILOT_CALIBRATION**: real-HE calibration/feasibility evidence.
- **HEIR_BOUNDARY_STRESS_DIAGNOSTIC**: deterministic mechanistic stress evidence around operational boundaries. It is intentionally enriched for fragile cases and must not be interpreted as a real-world disagreement rate.
- **HEIR_REAL_HE_CONFIRMATORY**: the only locked-test evidence for H1–H4.

The v1.3 locked test is the **original v1.0 held-out cohort**, reproduced by frozen split hashes. It is not an external validation cohort. Do not describe it as an independent external sample.

For zero disagreements, report the exact one-sided upper confidence bound. With only 114 locked-test observations, emphasize the corresponding rare-event resolution limit rather than writing that HE is universally decision invariant.

For H1, report the strict coexistence flag (zero class disagreements plus at least one operational disagreement) and separately report aggregate binary accuracy difference and classification disagreement rate. Do not invent an application-independent non-inferiority margin after seeing the test.

For H2, use the q=0.10 boundary definition as primary, report Holm-adjusted p-values across the two configurations, and present q=0.05/q=0.20 only as sensitivity analyses.

For H3, state explicitly that `error / operational_margin` is a mechanistically informed diagnostic score whose relationship to boundary crossing is partly structural. Its bootstrap AUC-difference interval is descriptive/diagnostic; no reject/accept multiplicity claim is made.

For H4, the primary endpoint is the paired difference in **incremental HE absolute error relative to the float32-export plaintext computation**. Total float64-to-HE error is secondary because it includes a common float32 serialization component.

**Multiplicity scope:** H2 and H4 are separately pre-specified confirmatory endpoints with different estimands. H2 controls family-wise error across its two configuration-specific Fisher tests using Holm. H3 is diagnostic. The study does **not** claim omnibus FWER control across the heterogeneous H1–H4 collection, so conclusions must remain endpoint-specific.

For the boundary-stress analysis, report both all constructed points and the pre-specified **plausible_joint_support** subset. The plausibility subset is defined using calibration-only marginal ranges, Ledoit-Wolf Mahalanobis distance, and kNN distance; it still does not become population evidence.

External validity remains limited by the use of one benchmark dataset and one affine logistic workload. A separate external cohort/workload should be treated as replication rather than retrofitted into this confirmatory family.

Before public release, populate `REPOSITORY_URL` and `LICENSE_CODE`; they are intentionally not inferred by the notebook.


## Source notes for infrastructure

The HEIR/OpenFHE infrastructure remains pinned to the same compiler release and source commit that completed the v1.0 real-HE pilot.

The CKKS parameter-generation interface supports explicit `min-slot-count`, `first-mod-bits`, and `scaling-mod-bits`; the study records the actual generated/runtime parameters rather than inferring them from configuration names. The OpenFHE build continues to use its installed-package CMake integration template.

The v1.3 robustness changes do not claim that lower modulus-bit settings automatically imply a specific security or precision level. Security metadata are reported only when explicitly present in generated/runtime evidence, and the primary H4 numerical comparison is empirical.
